<a href="https://colab.research.google.com/github/ShubhankarSanyal98/Tiny-ML/blob/main/Copy_of_Welcome_to_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

tiny ml project


In [ ]:
"""
TinyML HAR Comparative Evaluation Pipeline — Colab-ready, single file.
========================================================================
Towards TinyML Deployment: A Comparative Evaluation of Lightweight
Machine Learning Algorithms for Human Activity Recognition.

HOW TO RUN IN GOOGLE COLAB
---------------------------
1. Upload this file to Colab (or paste its contents into a cell) and run:
       !pip install -q memory_profiler xgboost seaborn
       !python har_tinyml.py
   or, if you pasted into a notebook cell, just run the cell — it will
   execute main() automatically when __name__ == "__main__" is satisfied
   (Colab cells run as __main__ too, so `python har_tinyml.py` and
   "run cell" behave the same way).

2. The script AUTO-DOWNLOADS the official UCI HAR Dataset the first time
   it runs (needs internet, which Colab has by default) and unzips it
   into ./data/UCI HAR Dataset/. No manual upload needed.

3. Outputs are written under ./figures, ./tables, ./results, ./models
   — in Colab, browse them in the Files pane on the left, or zip and
   download with:
       !zip -r outputs.zip figures tables results models
       from google.colab import files
       files.download('outputs.zip')

COMMAND-LINE FLAGS (also usable as `sys.argv` if pasted into a cell)
----------------------------------------------------------------------
    --data-dir DIR   Folder that contains (or will contain) 'UCI HAR Dataset'
    --fast           Tiny hyperparameter grids, quick smoke test
    --no-tune        Skip GridSearchCV, use default hyperparameters
    --cv N           Cross-validation folds (default 5)
    --no-download    Don't auto-download; expects data already in --data-dir
"""

import os
import sys
import io
import json
import time
import tempfile
import zipfile
import argparse
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from memory_profiler import memory_usage
    MEMORY_PROFILER_AVAILABLE = True
except ImportError:
    MEMORY_PROFILER_AVAILABLE = False

    def memory_usage(func, interval=0.01, max_usage=True):
        """Fallback if memory_profiler isn't installed: coarse peak-RSS via
        the resource module (Linux/Mac only)."""
        import resource
        func()
        peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return peak_kb / 1024  # MiB approx on Linux

warnings.filterwarnings("ignore")

RESULTS_DIR = "results"
FIGURES_DIR = "figures"
TABLES_DIR = "tables"
MODELS_DIR = "models"

UCI_HAR_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]

ACTIVITY_MAP = {
    1: "WALKING",
    2: "WALKING_UPSTAIRS",
    3: "WALKING_DOWNSTAIRS",
    4: "SITTING",
    5: "STANDING",
    6: "LAYING",
}

INTERPRETABILITY_RATING = {
    "Logistic Regression": 5,
    "Decision Tree": 5,
    "Gaussian Naive Bayes": 4,
    "KNN": 3,
    "Linear SVM": 3,
    "Random Forest": 2,
    "XGBoost": 1,
}


# ---------------------------------------------------------------------------
# 0. DATASET DOWNLOAD
# ---------------------------------------------------------------------------

def _find_dataset_root(search_dir):
    """Walk search_dir looking for the folder that actually contains
    features.txt + train/ + test/ (the real dataset root), regardless of
    what the containing folder happens to be named."""
    for root, dirs, files in os.walk(search_dir):
        if "features.txt" in files and "train" in dirs and "test" in dirs:
            return root
    return None


def download_uci_har(data_dir="data"):
    """Downloads + unzips the official UCI HAR Dataset if not already present.
    Handles mirrors that nest the real data inside an extra folder and/or
    inside a second, nested zip file."""
    target = os.path.join(data_dir, "UCI HAR Dataset")
    if os.path.isdir(target) and _find_dataset_root(target):
        print(f"Dataset already present at '{target}', skipping download.")
        return

    os.makedirs(data_dir, exist_ok=True)
    import urllib.request
    import shutil

    last_err = None
    for url in UCI_HAR_URLS:
        try:
            print(f"Downloading UCI HAR Dataset from {url} ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                raw = resp.read()

            extract_dir = os.path.join(data_dir, "_uci_har_download_tmp")
            if os.path.isdir(extract_dir):
                shutil.rmtree(extract_dir)
            os.makedirs(extract_dir, exist_ok=True)

            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                zf.extractall(extract_dir)
            print("Top-level download and extraction complete.")

            # Some mirrors wrap the real dataset in a second, nested zip
            # (e.g. "UCI HAR Dataset.zip" sitting inside the outer archive).
            found_root = _find_dataset_root(extract_dir)
            if not found_root:
                for root, _, files in os.walk(extract_dir):
                    for fname in files:
                        if fname.lower().endswith(".zip"):
                            nested_path = os.path.join(root, fname)
                            try:
                                with zipfile.ZipFile(nested_path) as nzf:
                                    nzf.extractall(root)
                                print(f"Extracted nested zip: {fname}")
                            except zipfile.BadZipFile:
                                continue
                found_root = _find_dataset_root(extract_dir)

            if not found_root:
                raise RuntimeError(
                    "Downloaded archive did not contain a recognizable "
                    "UCI HAR Dataset structure (features.txt + train/ + test/)."
                )

            if os.path.isdir(target):
                shutil.rmtree(target)
            shutil.move(found_root, target)
            shutil.rmtree(extract_dir, ignore_errors=True)

            print(f"Dataset ready at '{target}'.")
            return
        except Exception as e:
            last_err = e
            print(f"  Failed: {e}")
            continue

    raise RuntimeError(
        "Could not auto-download the UCI HAR Dataset "
        f"(last error: {last_err}).\n"
        "Please download it manually from "
        "https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones "
        f"and unzip it so that '{target}' exists, then re-run with --no-download."
    )


# ---------------------------------------------------------------------------
# 1. DATA LOADING
# ---------------------------------------------------------------------------

def _make_unique(names):
    seen = {}
    unique = []
    for n in names:
        if n in seen:
            seen[n] += 1
            unique.append(f"{n}_{seen[n]}")
        else:
            seen[n] = 0
            unique.append(n)
    return unique


def load_feature_names(dataset_root):
    features_path = os.path.join(dataset_root, "features.txt")
    features = pd.read_csv(features_path, sep=r"\s+", header=None, names=["idx", "name"])
    return _make_unique(features["name"].tolist())


def load_activity_labels(dataset_root):
    path = os.path.join(dataset_root, "activity_labels.txt")
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["id", "activity"])
    return dict(zip(df["id"], df["activity"]))


def _load_split(dataset_root, split, feature_names):
    split_dir = os.path.join(dataset_root, split)
    X = pd.read_csv(os.path.join(split_dir, f"X_{split}.txt"), sep=r"\s+", header=None)
    X.columns = feature_names
    y = pd.read_csv(os.path.join(split_dir, f"y_{split}.txt"), header=None, names=["Activity_Id"])
    subject_path = os.path.join(split_dir, f"subject_{split}.txt")
    subjects = pd.read_csv(subject_path, header=None, names=["Subject"]) if os.path.exists(subject_path) else None
    return X, y, subjects


def load_uci_har(data_dir="data"):
    dataset_root = os.path.join(data_dir, "UCI HAR Dataset")
    if not os.path.isdir(dataset_root) or not os.path.exists(os.path.join(dataset_root, "features.txt")):
        # Fall back to searching for the real root in case of extra nesting
        found = _find_dataset_root(data_dir) if os.path.isdir(data_dir) else None
        if found:
            dataset_root = found
        else:
            raise FileNotFoundError(
                f"Could not find a valid UCI HAR Dataset under '{data_dir}'. Run without "
                "--no-download to auto-fetch it, or download manually from "
                "https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones"
            )

    feature_names = load_feature_names(dataset_root)
    activity_map = load_activity_labels(dataset_root)

    X_train, y_train, subj_train = _load_split(dataset_root, "train", feature_names)
    X_test, y_test, subj_test = _load_split(dataset_root, "test", feature_names)

    y_train_labels = y_train["Activity_Id"].map(activity_map)
    y_test_labels = y_test["Activity_Id"].map(activity_map)

    return {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train["Activity_Id"], "y_test": y_test["Activity_Id"],
        "y_train_labels": y_train_labels, "y_test_labels": y_test_labels,
        "feature_names": feature_names, "activity_map": activity_map,
        "subjects_train": subj_train, "subjects_test": subj_test,
    }


def verify_dimensions(data):
    print("X_train:", data["X_train"].shape)
    print("y_train:", data["y_train"].shape)
    print("X_test :", data["X_test"].shape)
    print("y_test :", data["y_test"].shape)
    assert data["X_train"].shape[0] == data["y_train"].shape[0]
    assert data["X_test"].shape[0] == data["y_test"].shape[0]
    assert data["X_train"].shape[1] == data["X_test"].shape[1]
    print("Dimension check passed.")


# ---------------------------------------------------------------------------
# 2. EDA
# ---------------------------------------------------------------------------

def dataset_summary_table(data):
    rows = [
        {"Split": "Train", "Samples": data["X_train"].shape[0], "Features": data["X_train"].shape[1]},
        {"Split": "Test", "Samples": data["X_test"].shape[0], "Features": data["X_test"].shape[1]},
    ]
    return pd.DataFrame(rows)


def activity_distribution(y_labels, save_path=None):
    counts = y_labels.value_counts()
    plt.figure(figsize=(8, 5))
    sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Count")
    plt.title("Activity Distribution")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.close()
    return counts


def correlation_heatmap(X, top_n=30, save_path=None):
    subset = X.iloc[:, :top_n]
    corr = subset.corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title(f"Feature Correlation Heatmap (first {top_n} features)")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.close()


def run_eda(data, figures_dir=FIGURES_DIR, tables_dir=TABLES_DIR):
    os.makedirs(figures_dir, exist_ok=True)
    os.makedirs(tables_dir, exist_ok=True)

    summary = dataset_summary_table(data)
    summary.to_csv(os.path.join(tables_dir, "table1_dataset_summary.csv"), index=False)

    missing = data["X_train"].isnull().sum()
    print(f"Missing values in training set: {int((missing > 0).sum())}")

    activity_distribution(data["y_train_labels"], save_path=os.path.join(figures_dir, "activity_distribution.png"))
    correlation_heatmap(data["X_train"], save_path=os.path.join(figures_dir, "correlation_heatmap.png"))

    stats = data["X_train"].describe().T
    stats.to_csv(os.path.join(tables_dir, "feature_summary_statistics.csv"))

    print("EDA complete. Figures and tables saved.")
    return summary


# ---------------------------------------------------------------------------
# 3. PREPROCESSING
# ---------------------------------------------------------------------------

def scale_features(X_train, X_test, save_path=None):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    if save_path:
        joblib.dump(scaler, save_path)
    return X_train_scaled, X_test_scaled, scaler


def encode_labels(y_train, y_test, save_path=None):
    encoder = LabelEncoder()
    y_train_enc = encoder.fit_transform(y_train)
    y_test_enc = encoder.transform(y_test)
    if save_path:
        joblib.dump(encoder, save_path)
    return y_train_enc, y_test_enc, encoder


def integrity_check(X_train, X_test, y_train, y_test):
    checks = {
        "no_nan_train": not bool(X_train.isnull().values.any()),
        "no_nan_test": not bool(X_test.isnull().values.any()),
        "row_match_train": len(X_train) == len(y_train),
        "row_match_test": len(X_test) == len(y_test),
        "same_feature_count": X_train.shape[1] == X_test.shape[1],
    }
    print("Integrity checks:", checks)
    if not all(checks.values()):
        raise ValueError("Integrity check failed. See details above.")
    return checks


# ---------------------------------------------------------------------------
# 4. MODELS + HYPERPARAMETER GRIDS
# ---------------------------------------------------------------------------

def get_models_and_grids(fast_mode=False):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "KNN": KNeighborsClassifier(),
        "Gaussian Naive Bayes": GaussianNB(),
        "Linear SVM": LinearSVC(max_iter=5000, random_state=42),
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBClassifier(eval_metric="mlogloss", random_state=42, n_jobs=-1)

    if fast_mode:
        param_grids = {
            "Logistic Regression": {"C": [1.0]},
            "Decision Tree": {"max_depth": [10]},
            "Random Forest": {"n_estimators": [50]},
            "KNN": {"n_neighbors": [5]},
            "Gaussian Naive Bayes": {},
            "Linear SVM": {"C": [1.0]},
            "XGBoost": {"n_estimators": [50], "max_depth": [4]},
        }
    else:
        param_grids = {
            "Logistic Regression": {"C": [0.01, 0.1, 1, 10]},
            "Decision Tree": {"max_depth": [5, 10, 20, None], "min_samples_split": [2, 5, 10]},
            "Random Forest": {"n_estimators": [50, 100, 200], "max_depth": [10, 20, None]},
            "KNN": {"n_neighbors": [3, 5, 7, 9], "weights": ["uniform", "distance"]},
            "Gaussian Naive Bayes": {"var_smoothing": [1e-9, 1e-8, 1e-7]},
            "Linear SVM": {"C": [0.01, 0.1, 1, 10]},
            "XGBoost": {"n_estimators": [100, 200], "max_depth": [3, 5, 7], "learning_rate": [0.05, 0.1]},
        }

    param_grids = {k: v for k, v in param_grids.items() if k in models}
    return models, param_grids


# ---------------------------------------------------------------------------
# 5. TINYML-ORIENTED METRICS (the novelty)
# ---------------------------------------------------------------------------

def measure_training_time(model, X_train, y_train):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    return model, time.perf_counter() - start


def measure_prediction_time(model, X_test, n_repeats=5):
    single_sample = X_test[:1]
    latencies = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        model.predict(single_sample)
        latencies.append(time.perf_counter() - start)
    single_latency_ms = float(np.mean(latencies)) * 1000

    start = time.perf_counter()
    model.predict(X_test)
    batch_elapsed = time.perf_counter() - start
    throughput = X_test.shape[0] / batch_elapsed if batch_elapsed > 0 else float("inf")
    return single_latency_ms, throughput


def measure_model_size(model, tmp_dir=MODELS_DIR):
    os.makedirs(tmp_dir, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=tmp_dir, suffix=".joblib", delete=False) as tmp:
        joblib.dump(model, tmp.name)
        size_kb = os.path.getsize(tmp.name) / 1024
        tmp_path = tmp.name
    return size_kb, tmp_path


def measure_peak_memory(model, X_test):
    def _predict():
        model.predict(X_test)
    return float(memory_usage(_predict, interval=0.01, max_usage=True))


def estimate_model_complexity(model_name, model):
    complexity = {}
    try:
        if model_name in ("Logistic Regression", "Linear SVM"):
            complexity["n_parameters"] = int(np.prod(model.coef_.shape) + model.intercept_.shape[0])
        elif model_name == "Decision Tree":
            complexity["n_nodes"] = int(model.tree_.node_count)
            complexity["max_depth"] = int(model.get_depth())
        elif model_name == "Random Forest":
            complexity["n_trees"] = int(model.n_estimators)
            complexity["total_nodes"] = int(sum(t.tree_.node_count for t in model.estimators_))
        elif model_name == "KNN":
            complexity["n_stored_samples"] = int(model.n_samples_fit_)
        elif model_name == "Gaussian Naive Bayes":
            complexity["n_parameters"] = int(model.theta_.size + model.var_.size)
        elif model_name == "XGBoost":
            booster = model.get_booster()
            complexity["n_trees"] = int(booster.trees_to_dataframe()["Tree"].nunique())
        else:
            complexity["n_parameters"] = "n/a"
    except Exception as e:
        complexity["error"] = str(e)
    return complexity


def estimate_flash_ram(size_kb, peak_memory_mib):
    estimated_flash_kb = size_kb
    estimated_ram_kb = max(size_kb * 0.5, 1.0)
    return estimated_flash_kb, estimated_ram_kb


def deployment_friendliness_score(size_kb, single_latency_ms, interpretability):
    size_score = max(0, 100 - (size_kb / 5))
    latency_score = max(0, 100 - (single_latency_ms * 10))
    interp_score = (interpretability / 5) * 100
    score = 0.5 * size_score + 0.3 * latency_score + 0.2 * interp_score
    return round(max(0, min(100, score)), 2)


def evaluate_tinyml_metrics(model_name, model, X_train, y_train, X_test, models_dir=MODELS_DIR):
    fitted_model, train_time = measure_training_time(model, X_train, y_train)
    single_latency_ms, throughput = measure_prediction_time(fitted_model, X_test)
    size_kb, tmp_path = measure_model_size(fitted_model, tmp_dir=models_dir)
    peak_memory_mib = measure_peak_memory(fitted_model, X_test)
    complexity = estimate_model_complexity(model_name, fitted_model)
    flash_kb, ram_kb = estimate_flash_ram(size_kb, peak_memory_mib)
    interpretability = INTERPRETABILITY_RATING.get(model_name, 3)
    friendliness = deployment_friendliness_score(size_kb, single_latency_ms, interpretability)

    try:
        os.remove(tmp_path)
    except OSError:
        pass

    metrics = {
        "Model": model_name,
        "Training Time (s)": round(train_time, 4),
        "Single-Sample Latency (ms)": round(single_latency_ms, 4),
        "Batch Throughput (samples/s)": round(throughput, 2),
        "Serialized Model Size (KB)": round(size_kb, 2),
        "Peak Memory - Python process (MiB)": round(peak_memory_mib, 2),
        "Estimated Flash Usage (KB)": round(flash_kb, 2),
        "Estimated RAM Usage (KB)": round(ram_kb, 2),
        "Model Complexity": complexity,
        "Interpretability (1-5)": interpretability,
        "Deployment Friendliness (0-100)": friendliness,
    }
    return fitted_model, metrics


# ---------------------------------------------------------------------------
# 6. CLASSIFICATION EVALUATION
# ---------------------------------------------------------------------------

def evaluate_classifier(model, X_test, y_test, average="weighted"):
    y_pred = model.predict(X_test)
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average=average, zero_division=0),
        "Recall": recall_score(y_test, y_pred, average=average, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average=average, zero_division=0),
    }
    return {k: round(v, 4) for k, v in metrics.items()}, y_pred


def save_classification_report(model_name, y_test, y_pred, label_names, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    report = classification_report(y_test, y_pred, target_names=label_names, output_dict=True, zero_division=0)
    path = os.path.join(out_dir, f"{model_name.replace(' ', '_')}_report.json")
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
    return report


def plot_confusion_matrix(model_name, y_test, y_pred, label_names, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    path = os.path.join(out_dir, f"confusion_matrix_{model_name.replace(' ', '_')}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    return path


def run_cross_validation(model, X, y, cv=5):
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    return {"cv_mean_accuracy": round(scores.mean(), 4), "cv_std": round(scores.std(), 4)}


# ---------------------------------------------------------------------------
# 7. COMPARISON VISUALIZATIONS
# ---------------------------------------------------------------------------

def bar_comparison(df, metric_col, title, ylabel, out_path, ascending=False):
    plt.figure(figsize=(9, 5))
    sorted_df = df.sort_values(metric_col, ascending=ascending)
    bars = plt.bar(sorted_df["Model"], sorted_df[metric_col], color="#4C72B0")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, h, f"{h:.3g}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def radar_chart(df, metrics, out_path, title="Model Comparison Radar Chart"):
    norm_df = df.copy()
    for m in metrics:
        col = norm_df[m].astype(float)
        rng = col.max() - col.min()
        norm_df[m] = (col - col.min()) / rng if rng > 0 else 0.5

    labels = metrics
    n = len(labels)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for _, row in norm_df.iterrows():
        values = row[metrics].tolist()
        values += values[:1]
        ax.plot(angles, values, linewidth=1.5, label=row["Model"])
        ax.fill(angles, values, alpha=0.05)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_title(title)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def generate_all_visualizations(results_df, figures_dir=FIGURES_DIR):
    os.makedirs(figures_dir, exist_ok=True)

    bar_comparison(results_df, "Accuracy", "Accuracy Comparison", "Accuracy",
                    os.path.join(figures_dir, "accuracy_comparison.png"), ascending=False)
    bar_comparison(results_df, "F1-score", "F1-score Comparison", "F1-score",
                    os.path.join(figures_dir, "f1_comparison.png"), ascending=False)
    bar_comparison(results_df, "Precision", "Precision Comparison", "Precision",
                    os.path.join(figures_dir, "precision_comparison.png"), ascending=False)
    bar_comparison(results_df, "Recall", "Recall Comparison", "Recall",
                    os.path.join(figures_dir, "recall_comparison.png"), ascending=False)
    bar_comparison(results_df, "Training Time (s)", "Training Time Comparison", "Seconds",
                    os.path.join(figures_dir, "training_time_comparison.png"), ascending=True)
    bar_comparison(results_df, "Single-Sample Latency (ms)", "Prediction Latency Comparison", "Milliseconds",
                    os.path.join(figures_dir, "prediction_time_comparison.png"), ascending=True)
    bar_comparison(results_df, "Serialized Model Size (KB)", "Model Size Comparison", "KB",
                    os.path.join(figures_dir, "model_size_comparison.png"), ascending=True)
    bar_comparison(results_df, "Deployment Friendliness (0-100)", "Deployment Friendliness Ranking", "Score",
                    os.path.join(figures_dir, "deployment_friendliness.png"), ascending=False)

    radar_df = results_df.copy()
    radar_df["Speed (inv. latency)"] = -radar_df["Single-Sample Latency (ms)"]
    radar_df["Compactness (inv. size)"] = -radar_df["Serialized Model Size (KB)"]
    radar_metrics = ["Accuracy", "F1-score", "Speed (inv. latency)", "Compactness (inv. size)",
                      "Interpretability (1-5)"]
    radar_chart(radar_df, radar_metrics, os.path.join(figures_dir, "radar_chart.png"))

    print(f"All comparison visualizations saved to {figures_dir}/")


# ---------------------------------------------------------------------------
# 8. MAIN PIPELINE
# ---------------------------------------------------------------------------

def parse_args():
    p = argparse.ArgumentParser(description="TinyML HAR comparative evaluation pipeline")
    p.add_argument("--data-dir", default="data", help="Path containing (or to receive) 'UCI HAR Dataset'")
    p.add_argument("--fast", action="store_true", help="Use tiny hyperparameter grids for a quick run")
    p.add_argument("--no-tune", action="store_true", help="Skip GridSearchCV, use default hyperparameters")
    p.add_argument("--cv", type=int, default=5, help="Cross-validation folds")
    p.add_argument("--no-download", action="store_true", help="Don't auto-download the dataset")
    # parse_known_args so this also behaves in Jupyter/Colab cells that inject
    # extra kernel args (e.g. -f /root/.local/share/.../kernel.json)
    args, _ = p.parse_known_args()
    return args


def main():
    args = parse_args()

    for d in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR,
              os.path.join(RESULTS_DIR, "classification_reports"),
              os.path.join(RESULTS_DIR, "saved_models")]:
        os.makedirs(d, exist_ok=True)

    # 0. Download dataset if needed
    if not args.no_download:
        print("=" * 60)
        print("STEP 0: Ensuring dataset is available")
        print("=" * 60)
        download_uci_har(args.data_dir)

    # 1. Load Data
    print("\n" + "=" * 60)
    print("STEP 1: Loading data")
    print("=" * 60)
    data = load_uci_har(args.data_dir)
    verify_dimensions(data)

    # 2. EDA
    print("\n" + "=" * 60)
    print("STEP 2: Exploratory Data Analysis")
    print("=" * 60)
    run_eda(data)

    # 3. Preprocessing
    print("\n" + "=" * 60)
    print("STEP 3: Preprocessing")
    print("=" * 60)
    X_train_scaled, X_test_scaled, scaler = scale_features(
        data["X_train"], data["X_test"], save_path=os.path.join(MODELS_DIR, "scaler.joblib")
    )
    y_train_enc, y_test_enc, encoder = encode_labels(
        data["y_train_labels"], data["y_test_labels"], save_path=os.path.join(MODELS_DIR, "label_encoder.joblib")
    )
    integrity_check(data["X_train"], data["X_test"], data["y_train"], data["y_test"])
    label_names = list(encoder.classes_)

    # 4. Train + Tune + Evaluate + TinyML metrics
    print("\n" + "=" * 60)
    print("STEP 4: Training, tuning, and evaluating models")
    print("=" * 60)
    models, param_grids = get_models_and_grids(fast_mode=args.fast)

    all_rows = []
    param_table_rows = []

    for name, base_model in models.items():
        print(f"\n--- {name} ---")

        if args.no_tune:
            best_estimator = base_model
            best_params = base_model.get_params()
        else:
            grid = param_grids.get(name, {})
            if grid:
                search = GridSearchCV(base_model, grid, cv=args.cv, scoring="accuracy", n_jobs=-1)
                search.fit(X_train_scaled, y_train_enc)
                best_estimator = search.best_estimator_
                best_params = search.best_params_
                print(f"Best params: {best_params}")
            else:
                best_estimator = base_model
                best_params = base_model.get_params()

        clean_model = clone(best_estimator)

        fitted_model, tiny_metrics = evaluate_tinyml_metrics(
            name, clean_model, X_train_scaled, y_train_enc, X_test_scaled, models_dir=MODELS_DIR
        )

        perf_metrics, y_pred = evaluate_classifier(fitted_model, X_test_scaled, y_test_enc)
        save_classification_report(name, y_test_enc, y_pred, label_names,
                                    os.path.join(RESULTS_DIR, "classification_reports"))
        plot_confusion_matrix(name, y_test_enc, y_pred, label_names, FIGURES_DIR)
        cv_metrics = run_cross_validation(fitted_model, X_train_scaled, y_train_enc, cv=args.cv)

        row = {"Model": name, **perf_metrics, **cv_metrics, **tiny_metrics}
        row["Model Complexity"] = json.dumps(row["Model Complexity"])
        all_rows.append(row)

        model_path = os.path.join(RESULTS_DIR, "saved_models", f"{name.replace(' ', '_')}.joblib")
        joblib.dump(fitted_model, model_path)

        param_table_rows.append({"Model": name, "Best Parameters": json.dumps(best_params, default=str)})

        print(f"Accuracy={perf_metrics['Accuracy']}  F1={perf_metrics['F1-score']}  "
              f"Size={tiny_metrics['Serialized Model Size (KB)']}KB  "
              f"Latency={tiny_metrics['Single-Sample Latency (ms)']}ms  "
              f"Deploy-Score={tiny_metrics['Deployment Friendliness (0-100)']}")

    results_df = pd.DataFrame(all_rows)
    param_df = pd.DataFrame(param_table_rows)

    # 5. Save tables
    print("\n" + "=" * 60)
    print("STEP 5: Saving tables")
    print("=" * 60)
    results_df.to_csv(os.path.join(RESULTS_DIR, "comparison_results.csv"), index=False)
    param_df.to_csv(os.path.join(TABLES_DIR, "table2_algorithm_parameters.csv"), index=False)

    perf_cols = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "cv_mean_accuracy", "cv_std"]
    results_df[perf_cols].to_csv(os.path.join(TABLES_DIR, "table3_performance_metrics.csv"), index=False)

    tiny_cols = ["Model", "Training Time (s)", "Single-Sample Latency (ms)", "Batch Throughput (samples/s)",
                 "Serialized Model Size (KB)", "Peak Memory - Python process (MiB)",
                 "Estimated Flash Usage (KB)", "Estimated RAM Usage (KB)",
                 "Interpretability (1-5)", "Deployment Friendliness (0-100)"]
    results_df[tiny_cols].to_csv(os.path.join(TABLES_DIR, "table4_tinyml_metrics.csv"), index=False)

    # 6. Overall ranking + recommendation
    print("\n" + "=" * 60)
    print("STEP 6: Overall ranking and recommendation")
    print("=" * 60)
    ranking_df = results_df.copy()
    ranking_df["Composite Score"] = (
        0.6 * (ranking_df["F1-score"] * 100) + 0.4 * ranking_df["Deployment Friendliness (0-100)"]
    ).round(2)
    ranking_df = ranking_df.sort_values("Composite Score", ascending=False).reset_index(drop=True)
    ranking_df.insert(0, "Rank", ranking_df.index + 1)
    ranking_df[["Rank", "Model", "Accuracy", "F1-score", "Serialized Model Size (KB)",
                "Single-Sample Latency (ms)", "Deployment Friendliness (0-100)", "Composite Score"]].to_csv(
        os.path.join(TABLES_DIR, "table5_overall_ranking.csv"), index=False
    )

    best_model_name = ranking_df.iloc[0]["Model"]
    print(f"\nRECOMMENDED MODEL FOR TinyML DEPLOYMENT: {best_model_name}")
    print(ranking_df[["Rank", "Model", "Accuracy", "F1-score",
                       "Serialized Model Size (KB)", "Deployment Friendliness (0-100)",
                       "Composite Score"]].to_string(index=False))

    # 7. Visualizations
    print("\n" + "=" * 60)
    print("STEP 7: Generating comparison visualizations")
    print("=" * 60)
    generate_all_visualizations(results_df, figures_dir=FIGURES_DIR)

    print("\nPipeline complete. See results/, figures/, and tables/ for all outputs.")
    return results_df, ranking_df


if __name__ == "__main__":
    main()

STEP 0: Ensuring dataset is available
Dataset already present at 'data/UCI HAR Dataset', skipping download.

STEP 1: Loading data
X_train: (7352, 561)
y_train: (7352,)
X_test : (2947, 561)
y_test : (2947,)
Dimension check passed.

STEP 2: Exploratory Data Analysis
Missing values in training set: 0
EDA complete. Figures and tables saved.

STEP 3: Preprocessing
Integrity checks: {'no_nan_train': True, 'no_nan_test': True, 'row_match_train': True, 'row_match_test': True, 'same_feature_count': True}

STEP 4: Training, tuning, and evaluating models

--- Logistic Regression ---
Best params: {'C': 1}
Accuracy=0.9552  F1=0.9551  Size=27.21KB  Latency=0.5781ms  Deploy-Score=95.54

--- Decision Tree ---
Best params: {'max_depth': 10, 'min_samples_split': 5}
Accuracy=0.8666  F1=0.866  Size=24.4KB  Latency=0.174ms  Deploy-Score=97.04

--- Random Forest ---
Best params: {'max_depth': None, 'n_estimators': 50}
Accuracy=0.922  F1=0.9219  Size=2905.67KB  Latency=13.4346ms  Deploy-Score=8.0

--- KNN --

phase 2

In [ ]:
"""
TinyML HAR Comparative Evaluation Pipeline — Phase 1 + Phase 2 (Colab-ready, single file)
=============================================================================================
Towards TinyML Deployment: A Comparative Evaluation of Lightweight Machine
Learning Algorithms for Human Activity Recognition.

Phase 1 (baseline pipeline: UCI HAR)                -- included
Phase 2 (generalization: +WISDM, cross-dataset comparison) -- included
Phase 3 (robustness testing)                          -- NOT included (per scope)
Phase 5 (statistical validation)                      -- NOT included (per scope)
Phase 6 (paper writing)                               -- not code, out of scope

HOW TO RUN IN GOOGLE COLAB
---------------------------
    !pip install -q memory_profiler xgboost seaborn
    !python har_tinyml_phase2.py

The script auto-downloads BOTH datasets on first run:
  - UCI HAR Dataset  (archive.ics.uci.edu) — 561 pre-extracted features
  - WISDM v1.1 Activity Prediction (raw accelerometer -> windowed features,
    since WISDM ships as raw (x,y,z,t) signal rather than pre-extracted
    features; see extract_wisdm_features() for the one dataset-specific step)

If WISDM fails to download (the Fordham server is occasionally flaky), the
script logs a warning and continues with UCI HAR only — it does not crash.

Both datasets are run through the IDENTICAL downstream pipeline: same
algorithms, same hyperparameter grids, same GridSearchCV tuning strategy,
same classification metrics, same TinyML deployment metrics.

OUTPUTS (per dataset, under a folder named after the dataset)
----------------------------------------------------------------------------
figures/<dataset>/     - EDA plots, confusion matrices, comparison charts
tables/<dataset>/      - Table 1-5 (summary, params, performance, TinyML, ranking)
results/<dataset>/     - comparison_results.csv, classification_reports/, saved_models/

tables/cross_dataset/  - Table 10 series: UCI HAR vs. WISDM comparison
figures/cross_dataset/ - one grouped bar chart per metric (Accuracy, Precision,
                          Recall, F1, Training Time, Prediction Latency,
                          Model Size, Deployment Friendliness)

In Colab, browse these in the Files pane on the left, or zip and download:
    !zip -r outputs.zip figures tables results
    from google.colab import files
    files.download('outputs.zip')

COMMAND-LINE FLAGS
----------------------------------------------------------------------------
    --data-dir DIR       Folder that holds/receives raw datasets (default: data)
    --datasets LIST       Comma-separated: "uci,wisdm" (default), "uci", "wisdm"
    --fast                Tiny hyperparameter grids, quick smoke test
    --no-tune              Skip GridSearchCV, use default hyperparameters
    --cv N                  Cross-validation folds (default 5)
    --no-download           Don't auto-download; expects data already present
    --wisdm-window N        WISDM window size in samples (default 80, ~4s @20Hz)
    --wisdm-step N           WISDM window stride (default 40, 50% overlap)
"""

import os
import sys
import io
import json
import time
import tempfile
import zipfile
import tarfile
import argparse
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from memory_profiler import memory_usage
    MEMORY_PROFILER_AVAILABLE = True
except ImportError:
    MEMORY_PROFILER_AVAILABLE = False

    def memory_usage(func, interval=0.01, max_usage=True):
        """Fallback if memory_profiler isn't installed: coarse peak-RSS via
        the resource module (Linux/Mac only)."""
        import resource
        func()
        peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return peak_kb / 1024  # MiB approx on Linux

warnings.filterwarnings("ignore")

UCI_HAR_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]
WISDM_URLS = [
    "https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_ar_latest.tar.gz",
]

INTERPRETABILITY_RATING = {
    "Logistic Regression": 5,
    "Decision Tree": 5,
    "Gaussian Naive Bayes": 4,
    "KNN": 3,
    "Linear SVM": 3,
    "Random Forest": 2,
    "XGBoost": 1,
}

# Metrics requested for the Phase 2 cross-dataset comparison, and whether
# lower is better (used to pick sort order / bar coloring intent — kept
# simple here since we just plot raw values).
COMPARISON_METRICS = [
    ("Accuracy", "Accuracy"),
    ("Precision", "Precision"),
    ("Recall", "Recall"),
    ("F1-score", "F1-score"),
    ("Training Time (s)", "Training Time (s)"),
    ("Single-Sample Latency (ms)", "Prediction Latency (ms)"),
    ("Serialized Model Size (KB)", "Model Size (KB)"),
    ("Deployment Friendliness (0-100)", "Deployment Friendliness"),
]


# ===========================================================================
# 0. GENERIC ARCHIVE / SEARCH HELPERS
# ===========================================================================

def _find_dataset_root(search_dir):
    """Walk search_dir for the folder actually containing features.txt +
    train/ + test/ (UCI HAR's real root), regardless of enclosing folder name."""
    if not os.path.isdir(search_dir):
        return None
    for root, dirs, files in os.walk(search_dir):
        if "features.txt" in files and "train" in dirs and "test" in dirs:
            return root
    return None


def _find_file(search_dir, predicate):
    if not os.path.isdir(search_dir):
        return None
    for root, _, files in os.walk(search_dir):
        for fname in files:
            if predicate(fname):
                return os.path.join(root, fname)
    return None


# ===========================================================================
# 1. UCI HAR — DOWNLOAD + LOAD  (Phase 1)
# ===========================================================================

def download_uci_har(data_dir="data"):
    target = os.path.join(data_dir, "UCI HAR Dataset")
    if os.path.isdir(target) and _find_dataset_root(target):
        print(f"[UCI HAR] Already present at '{target}', skipping download.")
        return

    os.makedirs(data_dir, exist_ok=True)
    import urllib.request
    import shutil

    last_err = None
    for url in UCI_HAR_URLS:
        try:
            print(f"[UCI HAR] Downloading from {url} ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                raw = resp.read()

            extract_dir = os.path.join(data_dir, "_uci_har_download_tmp")
            if os.path.isdir(extract_dir):
                shutil.rmtree(extract_dir)
            os.makedirs(extract_dir, exist_ok=True)

            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                zf.extractall(extract_dir)
            print("[UCI HAR] Top-level extraction complete.")

            found_root = _find_dataset_root(extract_dir)
            if not found_root:
                # Some mirrors nest the real dataset inside a second zip file.
                for root, _, files in os.walk(extract_dir):
                    for fname in files:
                        if fname.lower().endswith(".zip"):
                            nested_path = os.path.join(root, fname)
                            try:
                                with zipfile.ZipFile(nested_path) as nzf:
                                    nzf.extractall(root)
                                print(f"[UCI HAR] Extracted nested zip: {fname}")
                            except zipfile.BadZipFile:
                                continue
                found_root = _find_dataset_root(extract_dir)

            if not found_root:
                raise RuntimeError("Archive did not contain a recognizable UCI HAR structure.")

            if os.path.isdir(target):
                shutil.rmtree(target)
            shutil.move(found_root, target)
            shutil.rmtree(extract_dir, ignore_errors=True)
            print(f"[UCI HAR] Dataset ready at '{target}'.")
            return
        except Exception as e:
            last_err = e
            print(f"[UCI HAR]   Failed: {e}")
            continue

    raise RuntimeError(
        f"[UCI HAR] Could not auto-download (last error: {last_err}). Download manually from "
        "https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones "
        f"and unzip so that '{target}' exists, then re-run with --no-download."
    )


def _make_unique(names):
    seen = {}
    unique = []
    for n in names:
        if n in seen:
            seen[n] += 1
            unique.append(f"{n}_{seen[n]}")
        else:
            seen[n] = 0
            unique.append(n)
    return unique


def load_uci_har(data_dir="data"):
    dataset_root = os.path.join(data_dir, "UCI HAR Dataset")
    if not os.path.isdir(dataset_root) or not os.path.exists(os.path.join(dataset_root, "features.txt")):
        found = _find_dataset_root(data_dir)
        if found:
            dataset_root = found
        else:
            raise FileNotFoundError(
                f"Could not find a valid UCI HAR Dataset under '{data_dir}'. Run without "
                "--no-download to auto-fetch it."
            )

    features = pd.read_csv(os.path.join(dataset_root, "features.txt"), sep=r"\s+", header=None,
                            names=["idx", "name"])
    feature_names = _make_unique(features["name"].tolist())

    labels_df = pd.read_csv(os.path.join(dataset_root, "activity_labels.txt"), sep=r"\s+",
                             header=None, names=["id", "activity"])
    activity_map = dict(zip(labels_df["id"], labels_df["activity"]))

    def _load_split(split):
        split_dir = os.path.join(dataset_root, split)
        X = pd.read_csv(os.path.join(split_dir, f"X_{split}.txt"), sep=r"\s+", header=None)
        X.columns = feature_names
        y = pd.read_csv(os.path.join(split_dir, f"y_{split}.txt"), header=None, names=["Activity_Id"])
        return X, y["Activity_Id"].map(activity_map)

    X_train, y_train_labels = _load_split("train")
    X_test, y_test_labels = _load_split("test")

    return {
        "X_train": X_train.reset_index(drop=True), "X_test": X_test.reset_index(drop=True),
        "y_train_labels": y_train_labels.reset_index(drop=True),
        "y_test_labels": y_test_labels.reset_index(drop=True),
        "feature_names": feature_names,
    }


# ===========================================================================
# 2. WISDM — DOWNLOAD + WINDOWED-FEATURE EXTRACTION + LOAD  (Phase 2)
# ===========================================================================

def download_wisdm(data_dir="data"):
    """Downloads the WISDM v1.1 raw accelerometer dataset. WISDM ships as
    raw (x,y,z,t) triaxial accelerometer readings, NOT pre-extracted
    features like UCI HAR — extract_wisdm_features() below builds a
    comparable feature table via sliding-window statistical features."""
    existing = _find_file(data_dir, lambda f: "raw" in f.lower() and f.lower().endswith(".txt")
                           and "wisdm" in f.lower())
    if existing:
        print(f"[WISDM] Already present at '{existing}', skipping download.")
        return existing

    os.makedirs(data_dir, exist_ok=True)
    import urllib.request

    last_err = None
    for url in WISDM_URLS:
        try:
            print(f"[WISDM] Downloading from {url} ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=90) as resp:
                raw = resp.read()

            extract_dir = os.path.join(data_dir, "WISDM_ar_v1.1")
            os.makedirs(extract_dir, exist_ok=True)

            with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tf:
                tf.extractall(extract_dir)
            print("[WISDM] Extraction complete.")

            raw_file = _find_file(extract_dir, lambda f: "raw" in f.lower() and f.lower().endswith(".txt"))
            if not raw_file:
                raise RuntimeError("Could not locate the raw .txt file inside the WISDM archive.")
            print(f"[WISDM] Dataset ready at '{raw_file}'.")
            return raw_file
        except Exception as e:
            last_err = e
            print(f"[WISDM]   Failed: {e}")
            continue

    print(
        f"[WISDM] WARNING: could not auto-download (last error: {last_err}). "
        "Continuing without WISDM — download manually from "
        "https://www.cis.fordham.edu/wisdm/dataset.php if you want Phase 2 "
        "generalization results."
    )
    return None


def parse_wisdm_raw(path):
    """WISDM's raw file is notoriously messy: records are ';'-terminated but
    also wrapped across newlines, with occasional malformed rows. Standard
    approach: strip newlines, split on ';', then split each record on ','."""
    with open(path, "r", errors="ignore") as f:
        content = f.read()
    content = content.replace("\n", "").replace("\r", "")
    records = content.split(";")

    rows = []
    for rec in records:
        rec = rec.strip().strip(",")
        if not rec:
            continue
        parts = rec.split(",")
        if len(parts) < 6:
            continue
        try:
            user = int(parts[0])
            activity = parts[1].strip()
            x, y, z = float(parts[3]), float(parts[4]), float(parts[5])
        except ValueError:
            continue
        rows.append((user, activity, x, y, z))

    return pd.DataFrame(rows, columns=["user", "activity", "x", "y", "z"])


def _make_windows(df, window_size, step):
    """Slide fixed-size windows within contiguous runs of the same
    (user, activity) so we never mix two activities in one window."""
    df = df.reset_index(drop=True)
    change = (df["user"] != df["user"].shift()) | (df["activity"] != df["activity"].shift())
    run_id = change.cumsum()
    windows = []
    for _, group in df.groupby(run_id):
        group = group.reset_index(drop=True)
        n = len(group)
        for start in range(0, max(n - window_size + 1, 0), step):
            windows.append(group.iloc[start:start + window_size])
    return windows


def _safe_corr(a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def _extract_window_features(seg):
    feats = {}
    for axis in ("x", "y", "z"):
        vals = seg[axis].values.astype(float)
        feats[f"{axis}_mean"] = float(np.mean(vals))
        feats[f"{axis}_std"] = float(np.std(vals))
        feats[f"{axis}_min"] = float(np.min(vals))
        feats[f"{axis}_max"] = float(np.max(vals))
        feats[f"{axis}_range"] = feats[f"{axis}_max"] - feats[f"{axis}_min"]
        feats[f"{axis}_energy"] = float(np.mean(vals ** 2))
        feats[f"{axis}_mad"] = float(np.mean(np.abs(vals - np.mean(vals))))
        feats[f"{axis}_iqr"] = float(np.percentile(vals, 75) - np.percentile(vals, 25))

    x, y, z = seg["x"].values.astype(float), seg["y"].values.astype(float), seg["z"].values.astype(float)
    feats["corr_xy"] = _safe_corr(x, y)
    feats["corr_xz"] = _safe_corr(x, z)
    feats["corr_yz"] = _safe_corr(y, z)

    mag = np.sqrt(x ** 2 + y ** 2 + z ** 2)
    feats["mag_mean"] = float(np.mean(mag))
    feats["mag_std"] = float(np.std(mag))
    feats["mag_max"] = float(np.max(mag))
    feats["mag_min"] = float(np.min(mag))

    feats["activity"] = seg["activity"].iloc[0]
    feats["user"] = seg["user"].iloc[0]
    return feats


def extract_wisdm_features(raw_df, window_size=80, step=40):
    windows = _make_windows(raw_df, window_size, step)
    rows = [_extract_window_features(w) for w in windows if len(w) == window_size]
    return pd.DataFrame(rows)


def load_wisdm(data_dir="data", window_size=80, step=40, test_size=0.2, random_state=42):
    raw_path = _find_file(data_dir, lambda f: "raw" in f.lower() and f.lower().endswith(".txt")
                           and "wisdm" in f.lower())
    if not raw_path:
        raise FileNotFoundError(
            f"Could not find a WISDM raw .txt file under '{data_dir}'. Run without "
            "--no-download to auto-fetch it, or download manually from "
            "https://www.cis.fordham.edu/wisdm/dataset.php"
        )

    print(f"[WISDM] Parsing raw accelerometer log: {raw_path}")
    raw_df = parse_wisdm_raw(raw_path)
    print(f"[WISDM] Parsed {len(raw_df)} raw readings across {raw_df['user'].nunique()} users, "
          f"{raw_df['activity'].nunique()} activities.")

    print(f"[WISDM] Extracting sliding-window statistical features "
          f"(window={window_size} samples, step={step})...")
    feat_df = extract_wisdm_features(raw_df, window_size=window_size, step=step)
    print(f"[WISDM] Built {len(feat_df)} feature windows, {feat_df.shape[1] - 2} features.")

    X = feat_df.drop(columns=["activity", "user"])
    y_labels = feat_df["activity"]

    X_train, X_test, y_train_labels, y_test_labels = train_test_split(
        X, y_labels, test_size=test_size, random_state=random_state, stratify=y_labels
    )

    return {
        "X_train": X_train.reset_index(drop=True), "X_test": X_test.reset_index(drop=True),
        "y_train_labels": y_train_labels.reset_index(drop=True),
        "y_test_labels": y_test_labels.reset_index(drop=True),
        "feature_names": list(X.columns),
    }


# ===========================================================================
# 3. GENERIC DATASET UTILITIES (dataset-agnostic from here on)
# ===========================================================================

def verify_dimensions(data, label=""):
    print(f"[{label}] X_train: {data['X_train'].shape}  y_train: {data['y_train_labels'].shape}")
    print(f"[{label}] X_test : {data['X_test'].shape}  y_test : {data['y_test_labels'].shape}")
    assert data["X_train"].shape[0] == data["y_train_labels"].shape[0]
    assert data["X_test"].shape[0] == data["y_test_labels"].shape[0]
    assert data["X_train"].shape[1] == data["X_test"].shape[1]
    print(f"[{label}] Dimension check passed.")


def run_eda(data, figures_dir, tables_dir, label=""):
    os.makedirs(figures_dir, exist_ok=True)
    os.makedirs(tables_dir, exist_ok=True)

    summary = pd.DataFrame([
        {"Split": "Train", "Samples": data["X_train"].shape[0], "Features": data["X_train"].shape[1]},
        {"Split": "Test", "Samples": data["X_test"].shape[0], "Features": data["X_test"].shape[1]},
    ])
    summary.to_csv(os.path.join(tables_dir, "table1_dataset_summary.csv"), index=False)

    missing = data["X_train"].isnull().sum()
    print(f"[{label}] Missing values in training set: {int((missing > 0).sum())}")

    counts = data["y_train_labels"].value_counts()
    plt.figure(figsize=(8, 5))
    sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Count")
    plt.title(f"Activity Distribution — {label}")
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, "activity_distribution.png"), dpi=150)
    plt.close()

    top_n = min(30, data["X_train"].shape[1])
    corr = data["X_train"].iloc[:, :top_n].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title(f"Feature Correlation Heatmap (first {top_n} features) — {label}")
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, "correlation_heatmap.png"), dpi=150)
    plt.close()

    data["X_train"].describe().T.to_csv(os.path.join(tables_dir, "feature_summary_statistics.csv"))
    print(f"[{label}] EDA complete.")
    return summary


def scale_features(X_train, X_test, save_path=None):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    if save_path:
        joblib.dump(scaler, save_path)
    return X_train_scaled, X_test_scaled, scaler


def encode_labels(y_train, y_test, save_path=None):
    encoder = LabelEncoder()
    y_train_enc = encoder.fit_transform(y_train)
    y_test_enc = encoder.transform(y_test)
    if save_path:
        joblib.dump(encoder, save_path)
    return y_train_enc, y_test_enc, encoder


def integrity_check(data, label=""):
    checks = {
        "no_nan_train": not bool(data["X_train"].isnull().values.any()),
        "no_nan_test": not bool(data["X_test"].isnull().values.any()),
        "row_match_train": len(data["X_train"]) == len(data["y_train_labels"]),
        "row_match_test": len(data["X_test"]) == len(data["y_test_labels"]),
        "same_feature_count": data["X_train"].shape[1] == data["X_test"].shape[1],
    }
    print(f"[{label}] Integrity checks: {checks}")
    if not all(checks.values()):
        raise ValueError(f"[{label}] Integrity check failed. See details above.")
    return checks


# ===========================================================================
# 4. MODELS + HYPERPARAMETER GRIDS
# ===========================================================================

def get_models_and_grids(fast_mode=False):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "KNN": KNeighborsClassifier(),
        "Gaussian Naive Bayes": GaussianNB(),
        "Linear SVM": LinearSVC(max_iter=5000, random_state=42),
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBClassifier(eval_metric="mlogloss", random_state=42, n_jobs=-1)

    if fast_mode:
        param_grids = {
            "Logistic Regression": {"C": [1.0]},
            "Decision Tree": {"max_depth": [10]},
            "Random Forest": {"n_estimators": [50]},
            "KNN": {"n_neighbors": [5]},
            "Gaussian Naive Bayes": {},
            "Linear SVM": {"C": [1.0]},
            "XGBoost": {"n_estimators": [50], "max_depth": [4]},
        }
    else:
        param_grids = {
            "Logistic Regression": {"C": [0.01, 0.1, 1, 10]},
            "Decision Tree": {"max_depth": [5, 10, 20, None], "min_samples_split": [2, 5, 10]},
            "Random Forest": {"n_estimators": [50, 100, 200], "max_depth": [10, 20, None]},
            "KNN": {"n_neighbors": [3, 5, 7, 9], "weights": ["uniform", "distance"]},
            "Gaussian Naive Bayes": {"var_smoothing": [1e-9, 1e-8, 1e-7]},
            "Linear SVM": {"C": [0.01, 0.1, 1, 10]},
            "XGBoost": {"n_estimators": [100, 200], "max_depth": [3, 5, 7], "learning_rate": [0.05, 0.1]},
        }

    param_grids = {k: v for k, v in param_grids.items() if k in models}
    return models, param_grids


# ===========================================================================
# 5. TINYML-ORIENTED METRICS (Phase 4, carried over into both datasets)
# ===========================================================================

def measure_training_time(model, X_train, y_train):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    return model, time.perf_counter() - start


def measure_prediction_time(model, X_test, n_repeats=5):
    single_sample = X_test[:1]
    latencies = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        model.predict(single_sample)
        latencies.append(time.perf_counter() - start)
    single_latency_ms = float(np.mean(latencies)) * 1000

    start = time.perf_counter()
    model.predict(X_test)
    batch_elapsed = time.perf_counter() - start
    throughput = X_test.shape[0] / batch_elapsed if batch_elapsed > 0 else float("inf")
    return single_latency_ms, throughput


def measure_model_size(model, tmp_dir):
    os.makedirs(tmp_dir, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=tmp_dir, suffix=".joblib", delete=False) as tmp:
        joblib.dump(model, tmp.name)
        size_kb = os.path.getsize(tmp.name) / 1024
        tmp_path = tmp.name
    return size_kb, tmp_path


def measure_peak_memory(model, X_test):
    def _predict():
        model.predict(X_test)
    return float(memory_usage(_predict, interval=0.01, max_usage=True))


def estimate_model_complexity(model_name, model):
    complexity = {}
    try:
        if model_name in ("Logistic Regression", "Linear SVM"):
            complexity["n_parameters"] = int(np.prod(model.coef_.shape) + model.intercept_.shape[0])
        elif model_name == "Decision Tree":
            complexity["n_nodes"] = int(model.tree_.node_count)
            complexity["max_depth"] = int(model.get_depth())
        elif model_name == "Random Forest":
            complexity["n_trees"] = int(model.n_estimators)
            complexity["total_nodes"] = int(sum(t.tree_.node_count for t in model.estimators_))
        elif model_name == "KNN":
            complexity["n_stored_samples"] = int(model.n_samples_fit_)
        elif model_name == "Gaussian Naive Bayes":
            complexity["n_parameters"] = int(model.theta_.size + model.var_.size)
        elif model_name == "XGBoost":
            booster = model.get_booster()
            complexity["n_trees"] = int(booster.trees_to_dataframe()["Tree"].nunique())
        else:
            complexity["n_parameters"] = "n/a"
    except Exception as e:
        complexity["error"] = str(e)
    return complexity


def deployment_friendliness_score(size_kb, single_latency_ms, interpretability):
    size_score = max(0, 100 - (size_kb / 5))
    latency_score = max(0, 100 - (single_latency_ms * 10))
    interp_score = (interpretability / 5) * 100
    score = 0.5 * size_score + 0.3 * latency_score + 0.2 * interp_score
    return round(max(0, min(100, score)), 2)


def evaluate_tinyml_metrics(model_name, model, X_train, y_train, X_test, models_dir):
    fitted_model, train_time = measure_training_time(model, X_train, y_train)
    single_latency_ms, throughput = measure_prediction_time(fitted_model, X_test)
    size_kb, tmp_path = measure_model_size(fitted_model, tmp_dir=models_dir)
    peak_memory_mib = measure_peak_memory(fitted_model, X_test)
    complexity = estimate_model_complexity(model_name, fitted_model)
    flash_kb = size_kb
    ram_kb = max(size_kb * 0.5, 1.0)
    interpretability = INTERPRETABILITY_RATING.get(model_name, 3)
    friendliness = deployment_friendliness_score(size_kb, single_latency_ms, interpretability)

    try:
        os.remove(tmp_path)
    except OSError:
        pass

    metrics = {
        "Model": model_name,
        "Training Time (s)": round(train_time, 4),
        "Single-Sample Latency (ms)": round(single_latency_ms, 4),
        "Batch Throughput (samples/s)": round(throughput, 2),
        "Serialized Model Size (KB)": round(size_kb, 2),
        "Peak Memory - Python process (MiB)": round(peak_memory_mib, 2),
        "Estimated Flash Usage (KB)": round(flash_kb, 2),
        "Estimated RAM Usage (KB)": round(ram_kb, 2),
        "Model Complexity": complexity,
        "Interpretability (1-5)": interpretability,
        "Deployment Friendliness (0-100)": friendliness,
    }
    return fitted_model, metrics


# ===========================================================================
# 6. CLASSIFICATION EVALUATION
# ===========================================================================

def evaluate_classifier(model, X_test, y_test, average="weighted"):
    y_pred = model.predict(X_test)
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average=average, zero_division=0),
        "Recall": recall_score(y_test, y_pred, average=average, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average=average, zero_division=0),
    }
    return {k: round(v, 4) for k, v in metrics.items()}, y_pred


def save_classification_report(model_name, y_test, y_pred, label_names, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    report = classification_report(y_test, y_pred, target_names=label_names, output_dict=True, zero_division=0)
    with open(os.path.join(out_dir, f"{model_name.replace(' ', '_')}_report.json"), "w") as f:
        json.dump(report, f, indent=2)
    return report


def plot_confusion_matrix(model_name, y_test, y_pred, label_names, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"confusion_matrix_{model_name.replace(' ', '_')}.png"), dpi=150)
    plt.close()


def run_cross_validation(model, X, y, cv=5):
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    return {"cv_mean_accuracy": round(scores.mean(), 4), "cv_std": round(scores.std(), 4)}


# ===========================================================================
# 7. COMPARISON VISUALIZATIONS (within a single dataset)
# ===========================================================================

def bar_comparison(df, metric_col, title, ylabel, out_path, ascending=False):
    plt.figure(figsize=(9, 5))
    sorted_df = df.sort_values(metric_col, ascending=ascending)
    bars = plt.bar(sorted_df["Model"], sorted_df[metric_col], color="#4C72B0")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, h, f"{h:.3g}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def radar_chart(df, metrics, out_path, title):
    norm_df = df.copy()
    for m in metrics:
        col = norm_df[m].astype(float)
        rng = col.max() - col.min()
        norm_df[m] = (col - col.min()) / rng if rng > 0 else 0.5

    n = len(metrics)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for _, row in norm_df.iterrows():
        values = row[metrics].tolist()
        values += values[:1]
        ax.plot(angles, values, linewidth=1.5, label=row["Model"])
        ax.fill(angles, values, alpha=0.05)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_title(title)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def generate_all_visualizations(results_df, figures_dir, label=""):
    os.makedirs(figures_dir, exist_ok=True)
    specs = [
        ("Accuracy", "Accuracy Comparison", "Accuracy", "accuracy_comparison.png", False),
        ("F1-score", "F1-score Comparison", "F1-score", "f1_comparison.png", False),
        ("Precision", "Precision Comparison", "Precision", "precision_comparison.png", False),
        ("Recall", "Recall Comparison", "Recall", "recall_comparison.png", False),
        ("Training Time (s)", "Training Time Comparison", "Seconds", "training_time_comparison.png", True),
        ("Single-Sample Latency (ms)", "Prediction Latency Comparison", "Milliseconds",
         "prediction_time_comparison.png", True),
        ("Serialized Model Size (KB)", "Model Size Comparison", "KB", "model_size_comparison.png", True),
        ("Deployment Friendliness (0-100)", "Deployment Friendliness Ranking", "Score",
         "deployment_friendliness.png", False),
    ]
    for col, title, ylabel, fname, asc in specs:
        bar_comparison(results_df, col, f"{title} — {label}", ylabel, os.path.join(figures_dir, fname), asc)

    radar_df = results_df.copy()
    radar_df["Speed (inv. latency)"] = -radar_df["Single-Sample Latency (ms)"]
    radar_df["Compactness (inv. size)"] = -radar_df["Serialized Model Size (KB)"]
    radar_metrics = ["Accuracy", "F1-score", "Speed (inv. latency)", "Compactness (inv. size)",
                      "Interpretability (1-5)"]
    radar_chart(radar_df, radar_metrics, os.path.join(figures_dir, "radar_chart.png"),
                f"Model Comparison Radar Chart — {label}")
    print(f"[{label}] All comparison visualizations saved to {figures_dir}/")


# ===========================================================================
# 8. PER-DATASET PIPELINE RUNNER (identical steps for UCI HAR and WISDM)
# ===========================================================================

def run_dataset_pipeline(dataset_name, data, args, out_root="."):
    print("\n" + "#" * 70)
    print(f"# DATASET: {dataset_name}")
    print("#" * 70)

    figures_dir = os.path.join(out_root, "figures", dataset_name)
    tables_dir = os.path.join(out_root, "tables", dataset_name)
    results_dir = os.path.join(out_root, "results", dataset_name)
    models_dir = os.path.join(out_root, "models", dataset_name)
    for d in [figures_dir, tables_dir, results_dir, models_dir,
              os.path.join(results_dir, "classification_reports"),
              os.path.join(results_dir, "saved_models")]:
        os.makedirs(d, exist_ok=True)

    verify_dimensions(data, label=dataset_name)
    run_eda(data, figures_dir, tables_dir, label=dataset_name)

    X_train_scaled, X_test_scaled, scaler = scale_features(
        data["X_train"], data["X_test"], save_path=os.path.join(models_dir, "scaler.joblib")
    )
    y_train_enc, y_test_enc, encoder = encode_labels(
        data["y_train_labels"], data["y_test_labels"], save_path=os.path.join(models_dir, "label_encoder.joblib")
    )
    integrity_check(data, label=dataset_name)
    label_names = list(encoder.classes_)

    models, param_grids = get_models_and_grids(fast_mode=args.fast)

    all_rows, param_table_rows = [], []

    for name, base_model in models.items():
        print(f"\n--- [{dataset_name}] {name} ---")

        if args.no_tune:
            best_estimator, best_params = base_model, base_model.get_params()
        else:
            grid = param_grids.get(name, {})
            if grid:
                search = GridSearchCV(base_model, grid, cv=args.cv, scoring="accuracy", n_jobs=-1)
                search.fit(X_train_scaled, y_train_enc)
                best_estimator, best_params = search.best_estimator_, search.best_params_
                print(f"Best params: {best_params}")
            else:
                best_estimator, best_params = base_model, base_model.get_params()

        clean_model = clone(best_estimator)
        fitted_model, tiny_metrics = evaluate_tinyml_metrics(
            name, clean_model, X_train_scaled, y_train_enc, X_test_scaled, models_dir=models_dir
        )

        perf_metrics, y_pred = evaluate_classifier(fitted_model, X_test_scaled, y_test_enc)
        save_classification_report(name, y_test_enc, y_pred, label_names,
                                    os.path.join(results_dir, "classification_reports"))
        plot_confusion_matrix(name, y_test_enc, y_pred, label_names, figures_dir)
        cv_metrics = run_cross_validation(fitted_model, X_train_scaled, y_train_enc, cv=args.cv)

        row = {"Model": name, **perf_metrics, **cv_metrics, **tiny_metrics}
        row["Model Complexity"] = json.dumps(row["Model Complexity"])
        all_rows.append(row)

        joblib.dump(fitted_model, os.path.join(results_dir, "saved_models", f"{name.replace(' ', '_')}.joblib"))
        param_table_rows.append({"Model": name, "Best Parameters": json.dumps(best_params, default=str)})

        print(f"Accuracy={perf_metrics['Accuracy']}  F1={perf_metrics['F1-score']}  "
              f"Size={tiny_metrics['Serialized Model Size (KB)']}KB  "
              f"Latency={tiny_metrics['Single-Sample Latency (ms)']}ms  "
              f"Deploy-Score={tiny_metrics['Deployment Friendliness (0-100)']}")

    results_df = pd.DataFrame(all_rows)
    pd.DataFrame(param_table_rows).to_csv(os.path.join(tables_dir, "table2_algorithm_parameters.csv"), index=False)
    results_df.to_csv(os.path.join(results_dir, "comparison_results.csv"), index=False)

    perf_cols = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "cv_mean_accuracy", "cv_std"]
    results_df[perf_cols].to_csv(os.path.join(tables_dir, "table3_performance_metrics.csv"), index=False)

    tiny_cols = ["Model", "Training Time (s)", "Single-Sample Latency (ms)", "Batch Throughput (samples/s)",
                 "Serialized Model Size (KB)", "Peak Memory - Python process (MiB)",
                 "Estimated Flash Usage (KB)", "Estimated RAM Usage (KB)",
                 "Interpretability (1-5)", "Deployment Friendliness (0-100)"]
    results_df[tiny_cols].to_csv(os.path.join(tables_dir, "table4_tinyml_metrics.csv"), index=False)

    ranking_df = results_df.copy()
    ranking_df["Composite Score"] = (
        0.6 * (ranking_df["F1-score"] * 100) + 0.4 * ranking_df["Deployment Friendliness (0-100)"]
    ).round(2)
    ranking_df = ranking_df.sort_values("Composite Score", ascending=False).reset_index(drop=True)
    ranking_df.insert(0, "Rank", ranking_df.index + 1)
    ranking_df[["Rank", "Model", "Accuracy", "F1-score", "Serialized Model Size (KB)",
                "Single-Sample Latency (ms)", "Deployment Friendliness (0-100)", "Composite Score"]].to_csv(
        os.path.join(tables_dir, "table5_overall_ranking.csv"), index=False
    )

    print(f"\n[{dataset_name}] RECOMMENDED MODEL: {ranking_df.iloc[0]['Model']}")
    print(ranking_df[["Rank", "Model", "Accuracy", "F1-score", "Deployment Friendliness (0-100)",
                       "Composite Score"]].to_string(index=False))

    generate_all_visualizations(results_df, figures_dir, label=dataset_name)

    return {"dataset_name": dataset_name, "results_df": results_df, "ranking_df": ranking_df}


# ===========================================================================
# 9. PHASE 2 — CROSS-DATASET COMPARISON
# ===========================================================================

def generate_comparison_tables(combined_df, tables_dir):
    os.makedirs(tables_dir, exist_ok=True)
    combined_df.to_csv(os.path.join(tables_dir, "table10_cross_dataset_comparison.csv"), index=False)

    wide_frames = []
    for col, _ in COMPARISON_METRICS:
        pivot = combined_df.pivot(index="Model", columns="Dataset", values=col)
        pivot.columns = [f"{ds} — {col}" for ds in pivot.columns]
        wide_frames.append(pivot)
    wide_df = pd.concat(wide_frames, axis=1).reset_index()
    wide_df.to_csv(os.path.join(tables_dir, "table10b_cross_dataset_wide.csv"), index=False)

    rank_rows = []
    for dataset in combined_df["Dataset"].unique():
        sub = combined_df[combined_df["Dataset"] == dataset].sort_values("Accuracy", ascending=False)
        rank_rows.append({"Dataset": dataset, "Top Model (by Accuracy)": sub.iloc[0]["Model"],
                           "Accuracy": sub.iloc[0]["Accuracy"]})
    pd.DataFrame(rank_rows).to_csv(os.path.join(tables_dir, "table10c_top_model_per_dataset.csv"), index=False)
    return wide_df


def generate_comparison_figures(combined_df, figures_dir):
    os.makedirs(figures_dir, exist_ok=True)
    datasets = combined_df["Dataset"].unique()

    for col, title in COMPARISON_METRICS:
        pivot = combined_df.pivot(index="Model", columns="Dataset", values=col)
        models = pivot.index.tolist()
        x = np.arange(len(models))
        width = 0.8 / max(len(datasets), 1)

        plt.figure(figsize=(10, 6))
        for i, dataset in enumerate(datasets):
            plt.bar(x + i * width, pivot[dataset], width=width, label=dataset)
        plt.xticks(x + width * (len(datasets) - 1) / 2, models, rotation=45, ha="right")
        plt.ylabel(title)
        plt.title(f"{title}: " + " vs. ".join(datasets))
        plt.legend()
        plt.tight_layout()
        fname = title.lower().replace(" ", "_").replace("(", "").replace(")", "")
        plt.savefig(os.path.join(figures_dir, f"cross_dataset_{fname}.png"), dpi=150)
        plt.close()

    print(f"Cross-dataset comparison figures saved to {figures_dir}/")


def compare_across_datasets(all_results, out_root="."):
    if len(all_results) < 2:
        print("\nOnly one dataset was evaluated — skipping Phase 2 cross-dataset comparison.")
        return

    print("\n" + "#" * 70)
    print("# PHASE 2: CROSS-DATASET COMPARISON (UCI HAR vs. WISDM)")
    print("#" * 70)

    merged_rows = []
    for res in all_results:
        for _, row in res["ranking_df"].iterrows():
            r = {"Dataset": res["dataset_name"], "Model": row["Model"]}
            for col, _ in COMPARISON_METRICS:
                r[col] = row[col]
            merged_rows.append(r)
    combined_df = pd.DataFrame(merged_rows)

    tables_dir = os.path.join(out_root, "tables", "cross_dataset")
    figures_dir = os.path.join(out_root, "figures", "cross_dataset")
    wide_df = generate_comparison_tables(combined_df, tables_dir)
    print(wide_df.to_string(index=False))

    generate_comparison_figures(combined_df, figures_dir)

    best_per_dataset = {res["dataset_name"]: res["ranking_df"].iloc[0]["Model"] for res in all_results}
    print("\nTop-ranked model per dataset:")
    for ds, m in best_per_dataset.items():
        print(f"  {ds}: {m}")
    if len(set(best_per_dataset.values())) == 1:
        winner = next(iter(best_per_dataset.values()))
        print(f"\n>> '{winner}' is the top-ranked model on EVERY evaluated dataset — "
              "early evidence the conclusion generalizes beyond a single dataset "
              "(formal statistical validation is Phase 5, not run here).")
    else:
        print("\n>> The top-ranked model differs across datasets — report both results "
              "transparently; see table10b_cross_dataset_wide.csv for the full breakdown.")

    print(f"\nCross-dataset comparison saved to {tables_dir}/ and {figures_dir}/")


# ===========================================================================
# 10. MAIN
# ===========================================================================

def parse_args():
    p = argparse.ArgumentParser(description="TinyML HAR pipeline — Phase 1 + Phase 2 (multi-dataset)")
    p.add_argument("--data-dir", default="data")
    p.add_argument("--datasets", default="uci,wisdm", help="Comma-separated: uci,wisdm")
    p.add_argument("--fast", action="store_true")
    p.add_argument("--no-tune", action="store_true")
    p.add_argument("--cv", type=int, default=5)
    p.add_argument("--no-download", action="store_true")
    p.add_argument("--wisdm-window", type=int, default=80)
    p.add_argument("--wisdm-step", type=int, default=40)
    # parse_known_args so this also behaves in Jupyter/Colab cells that inject
    # extra kernel args (e.g. -f /root/.local/share/.../kernel.json)
    args, _ = p.parse_known_args()
    return args


def main():
    args = parse_args()
    requested = [d.strip().lower() for d in args.datasets.split(",") if d.strip()]

    all_results = []

    if "uci" in requested:
        try:
            if not args.no_download:
                print("=" * 60)
                print("Ensuring UCI HAR dataset is available")
                print("=" * 60)
                download_uci_har(args.data_dir)
            data_uci = load_uci_har(args.data_dir)
            res = run_dataset_pipeline("UCI_HAR", data_uci, args)
            all_results.append(res)
        except Exception as e:
            print(f"\n[UCI HAR] Skipping dataset due to error: {e}")

    if "wisdm" in requested:
        try:
            if not args.no_download:
                print("\n" + "=" * 60)
                print("Ensuring WISDM dataset is available")
                print("=" * 60)
                download_wisdm(args.data_dir)
            data_wisdm = load_wisdm(args.data_dir, window_size=args.wisdm_window, step=args.wisdm_step)
            res = run_dataset_pipeline("WISDM", data_wisdm, args)
            all_results.append(res)
        except Exception as e:
            print(f"\n[WISDM] Skipping dataset due to error: {e}")

    if not all_results:
        print("\nNo datasets could be loaded. Nothing to do.")
        return

    compare_across_datasets(all_results)

    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE (Phase 1 + Phase 2)")
    print("=" * 70)
    print("See figures/<dataset>, tables/<dataset>, results/<dataset> for per-dataset outputs,")
    print("and tables/cross_dataset/, figures/cross_dataset/ for the Phase 2 comparison.")
    print("\nStopping here per scope — robustness testing (Phase 3), statistical validation")
    print("(Phase 5), and paper writing (Phase 6) are not included in this script.")

    return all_results


if __name__ == "__main__":
    main()

Ensuring UCI HAR dataset is available
[UCI HAR] Downloading from https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip ...
[UCI HAR] Top-level extraction complete.
[UCI HAR] Extracted nested zip: UCI HAR Dataset.zip
[UCI HAR] Dataset ready at 'data/UCI HAR Dataset'.

######################################################################
# DATASET: UCI_HAR
######################################################################
[UCI_HAR] X_train: (7352, 561)  y_train: (7352,)
[UCI_HAR] X_test : (2947, 561)  y_test : (2947,)
[UCI_HAR] Dimension check passed.
[UCI_HAR] Missing values in training set: 0
[UCI_HAR] EDA complete.
[UCI_HAR] Integrity checks: {'no_nan_train': True, 'no_nan_test': True, 'row_match_train': True, 'row_match_test': True, 'same_feature_count': True}

--- [UCI_HAR] Logistic Regression ---
Best params: {'C': 1}
Accuracy=0.9552  F1=0.9551  Size=27.21KB  Latency=0.6664ms  Deploy-Score=95.28

--- [UCI_HAR] Decision Tree ---
Best par

In [5]:
"""
TinyML HAR Comparative Evaluation Pipeline — Phase 1 + Phase 2 (Colab-ready, single file)
=============================================================================================
Towards TinyML Deployment: A Comparative Evaluation of Lightweight Machine
Learning Algorithms for Human Activity Recognition.

Phase 1 (baseline pipeline: UCI HAR)                -- included
Phase 2 (generalization: +WISDM, cross-dataset comparison) -- included
Phase 3 (robustness testing)                          -- NOT included (per scope)
Phase 5 (statistical validation)                      -- NOT included (per scope)
Phase 6 (paper writing)                               -- not code, out of scope

HOW TO RUN IN GOOGLE COLAB
---------------------------
    !pip install -q memory_profiler xgboost seaborn
    !python har_tinyml_phase2.py

The script auto-downloads BOTH datasets on first run:
  - UCI HAR Dataset  (archive.ics.uci.edu) — 561 pre-extracted features
  - WISDM v1.1 Activity Prediction (raw accelerometer -> windowed features,
    since WISDM ships as raw (x,y,z,t) signal rather than pre-extracted
    features; see extract_wisdm_features() for the one dataset-specific step)

If WISDM fails to download (the Fordham server is occasionally flaky), the
script logs a warning and continues with UCI HAR only — it does not crash.

Both datasets are run through the IDENTICAL downstream pipeline: same
algorithms, same hyperparameter grids, same GridSearchCV tuning strategy,
same classification metrics, same TinyML deployment metrics.

OUTPUTS (per dataset, under a folder named after the dataset)
----------------------------------------------------------------------------
figures/<dataset>/     - EDA plots, confusion matrices, comparison charts
tables/<dataset>/      - Table 1-5 (summary, params, performance, TinyML, ranking)
results/<dataset>/     - comparison_results.csv, classification_reports/, saved_models/

tables/cross_dataset/  - Table 10 series: UCI HAR vs. WISDM comparison
figures/cross_dataset/ - one grouped bar chart per metric (Accuracy, Precision,
                          Recall, F1, Training Time, Prediction Latency,
                          Model Size, Deployment Friendliness)

In Colab, browse these in the Files pane on the left, or zip and download:
    !zip -r outputs.zip figures tables results
    from google.colab import files
    files.download('outputs.zip')

COMMAND-LINE FLAGS
----------------------------------------------------------------------------
    --data-dir DIR       Folder that holds/receives raw datasets (default: data)
    --datasets LIST       Comma-separated: "uci,wisdm" (default), "uci", "wisdm"
    --fast                Tiny hyperparameter grids, quick smoke test
    --no-tune              Skip GridSearchCV, use default hyperparameters
    --cv N                  Cross-validation folds (default 5)
    --no-download           Don't auto-download; expects data already present
    --wisdm-window N        WISDM window size in samples (default 80, ~4s @20Hz)
    --wisdm-step N           WISDM window stride (default 40, 50% overlap)
"""

import os
import sys
import io
import json
import time
import tempfile
import zipfile
import tarfile
import argparse
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from memory_profiler import memory_usage
    MEMORY_PROFILER_AVAILABLE = True
except ImportError:
    MEMORY_PROFILER_AVAILABLE = False

    def memory_usage(func, interval=0.01, max_usage=True):
        """Fallback if memory_profiler isn't installed: coarse peak-RSS via
        the resource module (Linux/Mac only)."""
        import resource
        func()
        peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return peak_kb / 1024  # MiB approx on Linux

warnings.filterwarnings("ignore")

UCI_HAR_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]
WISDM_URLS = [
    "https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_ar_latest.tar.gz",
]

INTERPRETABILITY_RATING = {
    "Logistic Regression": 5,
    "Decision Tree": 5,
    "Gaussian Naive Bayes": 4,
    "KNN": 3,
    "Linear SVM": 3,
    "Random Forest": 2,
    "XGBoost": 1,
}

# Metrics requested for the Phase 2 cross-dataset comparison, and whether
# lower is better (used to pick sort order / bar coloring intent — kept
# simple here since we just plot raw values).
COMPARISON_METRICS = [
    ("Accuracy", "Accuracy"),
    ("Precision", "Precision"),
    ("Recall", "Recall"),
    ("F1-score", "F1-score"),
    ("Training Time (s)", "Training Time (s)"),
    ("Single-Sample Latency (ms)", "Prediction Latency (ms)"),
    ("Serialized Model Size (KB)", "Model Size (KB)"),
    ("Deployment Friendliness (0-100)", "Deployment Friendliness"),
]


# ===========================================================================
# 0. GENERIC ARCHIVE / SEARCH HELPERS
# ===========================================================================

def _find_dataset_root(search_dir):
    """Walk search_dir for the folder actually containing features.txt +
    train/ + test/ (UCI HAR's real root), regardless of enclosing folder name."""
    if not os.path.isdir(search_dir):
        return None
    for root, dirs, files in os.walk(search_dir):
        if "features.txt" in files and "train" in dirs and "test" in dirs:
            return root
    return None


def _find_file(search_dir, predicate):
    if not os.path.isdir(search_dir):
        return None
    for root, _, files in os.walk(search_dir):
        for fname in files:
            if predicate(fname):
                return os.path.join(root, fname)
    return None


def _is_wisdm_raw_file(fname):
    """Matches WISDM's actual data file (e.g. 'WISDM_ar_v1.1_raw.txt') while
    excluding companion docs that also contain 'raw' in the name, such as
    'WISDM_ar_v1.1_raw_about.txt'. Using endswith("raw.txt") rather than a
    substring check on "raw" is what makes this exclusion work: the "_about"
    file ends in "about.txt", not "raw.txt"."""
    f = fname.lower()
    return f.endswith("raw.txt") and "wisdm" in f


# ===========================================================================
# 1. UCI HAR — DOWNLOAD + LOAD  (Phase 1)
# ===========================================================================

def download_uci_har(data_dir="data"):
    target = os.path.join(data_dir, "UCI HAR Dataset")
    if os.path.isdir(target) and _find_dataset_root(target):
        print(f"[UCI HAR] Already present at '{target}', skipping download.")
        return

    os.makedirs(data_dir, exist_ok=True)
    import urllib.request
    import shutil

    last_err = None
    for url in UCI_HAR_URLS:
        try:
            print(f"[UCI HAR] Downloading from {url} ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                raw = resp.read()

            extract_dir = os.path.join(data_dir, "_uci_har_download_tmp")
            if os.path.isdir(extract_dir):
                shutil.rmtree(extract_dir)
            os.makedirs(extract_dir, exist_ok=True)

            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                zf.extractall(extract_dir)
            print("[UCI HAR] Top-level extraction complete.")

            found_root = _find_dataset_root(extract_dir)
            if not found_root:
                # Some mirrors nest the real dataset inside a second zip file.
                for root, _, files in os.walk(extract_dir):
                    for fname in files:
                        if fname.lower().endswith(".zip"):
                            nested_path = os.path.join(root, fname)
                            try:
                                with zipfile.ZipFile(nested_path) as nzf:
                                    nzf.extractall(root)
                                print(f"[UCI HAR] Extracted nested zip: {fname}")
                            except zipfile.BadZipFile:
                                continue
                found_root = _find_dataset_root(extract_dir)

            if not found_root:
                raise RuntimeError("Archive did not contain a recognizable UCI HAR structure.")

            if os.path.isdir(target):
                shutil.rmtree(target)
            shutil.move(found_root, target)
            shutil.rmtree(extract_dir, ignore_errors=True)
            print(f"[UCI HAR] Dataset ready at '{target}'.")
            return
        except Exception as e:
            last_err = e
            print(f"[UCI HAR]   Failed: {e}")
            continue

    raise RuntimeError(
        f"[UCI HAR] Could not auto-download (last error: {last_err}). Download manually from "
        "https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones "
        f"and unzip so that '{target}' exists, then re-run with --no-download."
    )


def _make_unique(names):
    seen = {}
    unique = []
    for n in names:
        if n in seen:
            seen[n] += 1
            unique.append(f"{n}_{seen[n]}")
        else:
            seen[n] = 0
            unique.append(n)
    return unique


def load_uci_har(data_dir="data"):
    dataset_root = os.path.join(data_dir, "UCI HAR Dataset")
    if not os.path.isdir(dataset_root) or not os.path.exists(os.path.join(dataset_root, "features.txt")):
        found = _find_dataset_root(data_dir)
        if found:
            dataset_root = found
        else:
            raise FileNotFoundError(
                f"Could not find a valid UCI HAR Dataset under '{data_dir}'. Run without "
                "--no-download to auto-fetch it."
            )

    features = pd.read_csv(os.path.join(dataset_root, "features.txt"), sep=r"\s+", header=None,
                            names=["idx", "name"])
    feature_names = _make_unique(features["name"].tolist())

    labels_df = pd.read_csv(os.path.join(dataset_root, "activity_labels.txt"), sep=r"\s+",
                             header=None, names=["id", "activity"])
    activity_map = dict(zip(labels_df["id"], labels_df["activity"]))

    def _load_split(split):
        split_dir = os.path.join(dataset_root, split)
        X = pd.read_csv(os.path.join(split_dir, f"X_{split}.txt"), sep=r"\s+", header=None)
        X.columns = feature_names
        y = pd.read_csv(os.path.join(split_dir, f"y_{split}.txt"), header=None, names=["Activity_Id"])
        return X, y["Activity_Id"].map(activity_map)

    X_train, y_train_labels = _load_split("train")
    X_test, y_test_labels = _load_split("test")

    return {
        "X_train": X_train.reset_index(drop=True), "X_test": X_test.reset_index(drop=True),
        "y_train_labels": y_train_labels.reset_index(drop=True),
        "y_test_labels": y_test_labels.reset_index(drop=True),
        "feature_names": feature_names,
    }


# ===========================================================================
# 2. WISDM — DOWNLOAD + WINDOWED-FEATURE EXTRACTION + LOAD  (Phase 2)
# ===========================================================================

def download_wisdm(data_dir="data"):
    """Downloads the WISDM v1.1 raw accelerometer dataset. WISDM ships as
    raw (x,y,z,t) triaxial accelerometer readings, NOT pre-extracted
    features like UCI HAR — extract_wisdm_features() below builds a
    comparable feature table via sliding-window statistical features."""
    existing = _find_file(data_dir, _is_wisdm_raw_file)
    if existing:
        print(f"[WISDM] Already present at '{existing}', skipping download.")
        return existing

    os.makedirs(data_dir, exist_ok=True)
    import urllib.request

    last_err = None
    for url in WISDM_URLS:
        try:
            print(f"[WISDM] Downloading from {url} ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=90) as resp:
                raw = resp.read()

            extract_dir = os.path.join(data_dir, "WISDM_ar_v1.1")
            os.makedirs(extract_dir, exist_ok=True)

            with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tf:
                tf.extractall(extract_dir)
            print("[WISDM] Extraction complete.")

            raw_file = _find_file(extract_dir, _is_wisdm_raw_file)
            if not raw_file:
                raise RuntimeError("Could not locate the raw .txt file inside the WISDM archive.")
            print(f"[WISDM] Dataset ready at '{raw_file}'.")
            return raw_file
        except Exception as e:
            last_err = e
            print(f"[WISDM]   Failed: {e}")
            continue

    print(
        f"[WISDM] WARNING: could not auto-download (last error: {last_err}). "
        "Continuing without WISDM — download manually from "
        "https://www.cis.fordham.edu/wisdm/dataset.php if you want Phase 2 "
        "generalization results."
    )
    return None


def parse_wisdm_raw(path):
    """WISDM's raw file is notoriously messy: records are ';'-terminated but
    also wrapped across newlines, with occasional malformed rows. Standard
    approach: strip newlines, split on ';', then split each record on ','."""
    with open(path, "r", errors="ignore") as f:
        content = f.read()
    content = content.replace("\n", "").replace("\r", "")
    records = content.split(";")

    rows = []
    for rec in records:
        rec = rec.strip().strip(",")
        if not rec:
            continue
        parts = rec.split(",")
        if len(parts) < 6:
            continue
        try:
            user = int(parts[0])
            activity = parts[1].strip()
            x, y, z = float(parts[3]), float(parts[4]), float(parts[5])
        except ValueError:
            continue
        rows.append((user, activity, x, y, z))

    return pd.DataFrame(rows, columns=["user", "activity", "x", "y", "z"])


def _make_windows(df, window_size, step):
    """Slide fixed-size windows within contiguous runs of the same
    (user, activity) so we never mix two activities in one window."""
    df = df.reset_index(drop=True)
    change = (df["user"] != df["user"].shift()) | (df["activity"] != df["activity"].shift())
    run_id = change.cumsum()
    windows = []
    for _, group in df.groupby(run_id):
        group = group.reset_index(drop=True)
        n = len(group)
        for start in range(0, max(n - window_size + 1, 0), step):
            windows.append(group.iloc[start:start + window_size])
    return windows


def _safe_corr(a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def _extract_window_features(seg):
    feats = {}
    for axis in ("x", "y", "z"):
        vals = seg[axis].values.astype(float)
        feats[f"{axis}_mean"] = float(np.mean(vals))
        feats[f"{axis}_std"] = float(np.std(vals))
        feats[f"{axis}_min"] = float(np.min(vals))
        feats[f"{axis}_max"] = float(np.max(vals))
        feats[f"{axis}_range"] = feats[f"{axis}_max"] - feats[f"{axis}_min"]
        feats[f"{axis}_energy"] = float(np.mean(vals ** 2))
        feats[f"{axis}_mad"] = float(np.mean(np.abs(vals - np.mean(vals))))
        feats[f"{axis}_iqr"] = float(np.percentile(vals, 75) - np.percentile(vals, 25))

    x, y, z = seg["x"].values.astype(float), seg["y"].values.astype(float), seg["z"].values.astype(float)
    feats["corr_xy"] = _safe_corr(x, y)
    feats["corr_xz"] = _safe_corr(x, z)
    feats["corr_yz"] = _safe_corr(y, z)

    mag = np.sqrt(x ** 2 + y ** 2 + z ** 2)
    feats["mag_mean"] = float(np.mean(mag))
    feats["mag_std"] = float(np.std(mag))
    feats["mag_max"] = float(np.max(mag))
    feats["mag_min"] = float(np.min(mag))

    feats["activity"] = seg["activity"].iloc[0]
    feats["user"] = seg["user"].iloc[0]
    return feats


def extract_wisdm_features(raw_df, window_size=80, step=40):
    windows = _make_windows(raw_df, window_size, step)
    rows = [_extract_window_features(w) for w in windows if len(w) == window_size]
    return pd.DataFrame(rows)


def load_wisdm(data_dir="data", window_size=80, step=40, test_size=0.2, random_state=42):
    raw_path = _find_file(data_dir, _is_wisdm_raw_file)
    if not raw_path:
        raise FileNotFoundError(
            f"Could not find a WISDM raw .txt file under '{data_dir}'. Run without "
            "--no-download to auto-fetch it, or download manually from "
            "https://www.cis.fordham.edu/wisdm/dataset.php"
        )

    print(f"[WISDM] Parsing raw accelerometer log: {raw_path}")
    raw_df = parse_wisdm_raw(raw_path)
    if len(raw_df) == 0:
        raise ValueError(
            f"Parsed 0 readings from '{raw_path}' — this usually means the wrong file was "
            "picked up (e.g. a readme/about file instead of the actual raw data file). "
            "Delete the data directory and re-run to force a fresh download."
        )
    print(f"[WISDM] Parsed {len(raw_df)} raw readings across {raw_df['user'].nunique()} users, "
          f"{raw_df['activity'].nunique()} activities.")

    print(f"[WISDM] Extracting sliding-window statistical features "
          f"(window={window_size} samples, step={step})...")
    feat_df = extract_wisdm_features(raw_df, window_size=window_size, step=step)
    if len(feat_df) == 0:
        raise ValueError(
            "No sliding windows could be extracted — check --wisdm-window/--wisdm-step "
            "against the amount of parsed data."
        )
    print(f"[WISDM] Built {len(feat_df)} feature windows, {feat_df.shape[1] - 2} features.")

    X = feat_df.drop(columns=["activity", "user"])
    y_labels = feat_df["activity"]

    X_train, X_test, y_train_labels, y_test_labels = train_test_split(
        X, y_labels, test_size=test_size, random_state=random_state, stratify=y_labels
    )

    return {
        "X_train": X_train.reset_index(drop=True), "X_test": X_test.reset_index(drop=True),
        "y_train_labels": y_train_labels.reset_index(drop=True),
        "y_test_labels": y_test_labels.reset_index(drop=True),
        "feature_names": list(X.columns),
    }


# ===========================================================================
# 3. GENERIC DATASET UTILITIES (dataset-agnostic from here on)
# ===========================================================================

def verify_dimensions(data, label=""):
    print(f"[{label}] X_train: {data['X_train'].shape}  y_train: {data['y_train_labels'].shape}")
    print(f"[{label}] X_test : {data['X_test'].shape}  y_test : {data['y_test_labels'].shape}")
    assert data["X_train"].shape[0] == data["y_train_labels"].shape[0]
    assert data["X_test"].shape[0] == data["y_test_labels"].shape[0]
    assert data["X_train"].shape[1] == data["X_test"].shape[1]
    print(f"[{label}] Dimension check passed.")


def run_eda(data, figures_dir, tables_dir, label=""):
    os.makedirs(figures_dir, exist_ok=True)
    os.makedirs(tables_dir, exist_ok=True)

    summary = pd.DataFrame([
        {"Split": "Train", "Samples": data["X_train"].shape[0], "Features": data["X_train"].shape[1]},
        {"Split": "Test", "Samples": data["X_test"].shape[0], "Features": data["X_test"].shape[1]},
    ])
    summary.to_csv(os.path.join(tables_dir, "table1_dataset_summary.csv"), index=False)

    missing = data["X_train"].isnull().sum()
    print(f"[{label}] Missing values in training set: {int((missing > 0).sum())}")

    counts = data["y_train_labels"].value_counts()
    plt.figure(figsize=(8, 5))
    sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Count")
    plt.title(f"Activity Distribution — {label}")
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, "activity_distribution.png"), dpi=150)
    plt.close()

    top_n = min(30, data["X_train"].shape[1])
    corr = data["X_train"].iloc[:, :top_n].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title(f"Feature Correlation Heatmap (first {top_n} features) — {label}")
    plt.tight_layout()
    plt.savefig(os.path.join(figures_dir, "correlation_heatmap.png"), dpi=150)
    plt.close()

    data["X_train"].describe().T.to_csv(os.path.join(tables_dir, "feature_summary_statistics.csv"))
    print(f"[{label}] EDA complete.")
    return summary


def scale_features(X_train, X_test, save_path=None):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    if save_path:
        joblib.dump(scaler, save_path)
    return X_train_scaled, X_test_scaled, scaler


def encode_labels(y_train, y_test, save_path=None):
    encoder = LabelEncoder()
    y_train_enc = encoder.fit_transform(y_train)
    y_test_enc = encoder.transform(y_test)
    if save_path:
        joblib.dump(encoder, save_path)
    return y_train_enc, y_test_enc, encoder


def integrity_check(data, label=""):
    checks = {
        "no_nan_train": not bool(data["X_train"].isnull().values.any()),
        "no_nan_test": not bool(data["X_test"].isnull().values.any()),
        "row_match_train": len(data["X_train"]) == len(data["y_train_labels"]),
        "row_match_test": len(data["X_test"]) == len(data["y_test_labels"]),
        "same_feature_count": data["X_train"].shape[1] == data["X_test"].shape[1],
    }
    print(f"[{label}] Integrity checks: {checks}")
    if not all(checks.values()):
        raise ValueError(f"[{label}] Integrity check failed. See details above.")
    return checks


# ===========================================================================
# 4. MODELS + HYPERPARAMETER GRIDS
# ===========================================================================

def get_models_and_grids(fast_mode=False):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "KNN": KNeighborsClassifier(),
        "Gaussian Naive Bayes": GaussianNB(),
        "Linear SVM": LinearSVC(max_iter=5000, random_state=42),
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBClassifier(eval_metric="mlogloss", random_state=42, n_jobs=-1)

    if fast_mode:
        param_grids = {
            "Logistic Regression": {"C": [1.0]},
            "Decision Tree": {"max_depth": [10]},
            "Random Forest": {"n_estimators": [50]},
            "KNN": {"n_neighbors": [5]},
            "Gaussian Naive Bayes": {},
            "Linear SVM": {"C": [1.0]},
            "XGBoost": {"n_estimators": [50], "max_depth": [4]},
        }
    else:
        param_grids = {
            "Logistic Regression": {"C": [0.01, 0.1, 1, 10]},
            "Decision Tree": {"max_depth": [5, 10, 20, None], "min_samples_split": [2, 5, 10]},
            "Random Forest": {"n_estimators": [50, 100, 200], "max_depth": [10, 20, None]},
            "KNN": {"n_neighbors": [3, 5, 7, 9], "weights": ["uniform", "distance"]},
            "Gaussian Naive Bayes": {"var_smoothing": [1e-9, 1e-8, 1e-7]},
            "Linear SVM": {"C": [0.01, 0.1, 1, 10]},
            "XGBoost": {"n_estimators": [100, 200], "max_depth": [3, 5, 7], "learning_rate": [0.05, 0.1]},
        }

    param_grids = {k: v for k, v in param_grids.items() if k in models}
    return models, param_grids


# ===========================================================================
# 5. TINYML-ORIENTED METRICS (Phase 4, carried over into both datasets)
# ===========================================================================

def measure_training_time(model, X_train, y_train):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    return model, time.perf_counter() - start


def measure_prediction_time(model, X_test, n_repeats=5):
    single_sample = X_test[:1]
    latencies = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        model.predict(single_sample)
        latencies.append(time.perf_counter() - start)
    single_latency_ms = float(np.mean(latencies)) * 1000

    start = time.perf_counter()
    model.predict(X_test)
    batch_elapsed = time.perf_counter() - start
    throughput = X_test.shape[0] / batch_elapsed if batch_elapsed > 0 else float("inf")
    return single_latency_ms, throughput


def measure_model_size(model, tmp_dir):
    os.makedirs(tmp_dir, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=tmp_dir, suffix=".joblib", delete=False) as tmp:
        joblib.dump(model, tmp.name)
        size_kb = os.path.getsize(tmp.name) / 1024
        tmp_path = tmp.name
    return size_kb, tmp_path


def measure_peak_memory(model, X_test):
    def _predict():
        model.predict(X_test)
    return float(memory_usage(_predict, interval=0.01, max_usage=True))


def estimate_model_complexity(model_name, model):
    complexity = {}
    try:
        if model_name in ("Logistic Regression", "Linear SVM"):
            complexity["n_parameters"] = int(np.prod(model.coef_.shape) + model.intercept_.shape[0])
        elif model_name == "Decision Tree":
            complexity["n_nodes"] = int(model.tree_.node_count)
            complexity["max_depth"] = int(model.get_depth())
        elif model_name == "Random Forest":
            complexity["n_trees"] = int(model.n_estimators)
            complexity["total_nodes"] = int(sum(t.tree_.node_count for t in model.estimators_))
        elif model_name == "KNN":
            complexity["n_stored_samples"] = int(model.n_samples_fit_)
        elif model_name == "Gaussian Naive Bayes":
            complexity["n_parameters"] = int(model.theta_.size + model.var_.size)
        elif model_name == "XGBoost":
            booster = model.get_booster()
            complexity["n_trees"] = int(booster.trees_to_dataframe()["Tree"].nunique())
        else:
            complexity["n_parameters"] = "n/a"
    except Exception as e:
        complexity["error"] = str(e)
    return complexity


def deployment_friendliness_score(size_kb, single_latency_ms, interpretability):
    size_score = max(0, 100 - (size_kb / 5))
    latency_score = max(0, 100 - (single_latency_ms * 10))
    interp_score = (interpretability / 5) * 100
    score = 0.5 * size_score + 0.3 * latency_score + 0.2 * interp_score
    return round(max(0, min(100, score)), 2)


def evaluate_tinyml_metrics(model_name, model, X_train, y_train, X_test, models_dir):
    fitted_model, train_time = measure_training_time(model, X_train, y_train)
    single_latency_ms, throughput = measure_prediction_time(fitted_model, X_test)
    size_kb, tmp_path = measure_model_size(fitted_model, tmp_dir=models_dir)
    peak_memory_mib = measure_peak_memory(fitted_model, X_test)
    complexity = estimate_model_complexity(model_name, fitted_model)
    flash_kb = size_kb
    ram_kb = max(size_kb * 0.5, 1.0)
    interpretability = INTERPRETABILITY_RATING.get(model_name, 3)
    friendliness = deployment_friendliness_score(size_kb, single_latency_ms, interpretability)

    try:
        os.remove(tmp_path)
    except OSError:
        pass

    metrics = {
        "Model": model_name,
        "Training Time (s)": round(train_time, 4),
        "Single-Sample Latency (ms)": round(single_latency_ms, 4),
        "Batch Throughput (samples/s)": round(throughput, 2),
        "Serialized Model Size (KB)": round(size_kb, 2),
        "Peak Memory - Python process (MiB)": round(peak_memory_mib, 2),
        "Estimated Flash Usage (KB)": round(flash_kb, 2),
        "Estimated RAM Usage (KB)": round(ram_kb, 2),
        "Model Complexity": complexity,
        "Interpretability (1-5)": interpretability,
        "Deployment Friendliness (0-100)": friendliness,
    }
    return fitted_model, metrics


# ===========================================================================
# 6. CLASSIFICATION EVALUATION
# ===========================================================================

def evaluate_classifier(model, X_test, y_test, average="weighted"):
    y_pred = model.predict(X_test)
    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average=average, zero_division=0),
        "Recall": recall_score(y_test, y_pred, average=average, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average=average, zero_division=0),
    }
    return {k: round(v, 4) for k, v in metrics.items()}, y_pred


def save_classification_report(model_name, y_test, y_pred, label_names, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    report = classification_report(y_test, y_pred, target_names=label_names, output_dict=True, zero_division=0)
    with open(os.path.join(out_dir, f"{model_name.replace(' ', '_')}_report.json"), "w") as f:
        json.dump(report, f, indent=2)
    return report


def plot_confusion_matrix(model_name, y_test, y_pred, label_names, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"confusion_matrix_{model_name.replace(' ', '_')}.png"), dpi=150)
    plt.close()


def run_cross_validation(model, X, y, cv=5):
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    return {"cv_mean_accuracy": round(scores.mean(), 4), "cv_std": round(scores.std(), 4)}


# ===========================================================================
# 7. COMPARISON VISUALIZATIONS (within a single dataset)
# ===========================================================================

def bar_comparison(df, metric_col, title, ylabel, out_path, ascending=False):
    plt.figure(figsize=(9, 5))
    sorted_df = df.sort_values(metric_col, ascending=ascending)
    bars = plt.bar(sorted_df["Model"], sorted_df[metric_col], color="#4C72B0")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, h, f"{h:.3g}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def radar_chart(df, metrics, out_path, title):
    norm_df = df.copy()
    for m in metrics:
        col = norm_df[m].astype(float)
        rng = col.max() - col.min()
        norm_df[m] = (col - col.min()) / rng if rng > 0 else 0.5

    n = len(metrics)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for _, row in norm_df.iterrows():
        values = row[metrics].tolist()
        values += values[:1]
        ax.plot(angles, values, linewidth=1.5, label=row["Model"])
        ax.fill(angles, values, alpha=0.05)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_title(title)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def generate_all_visualizations(results_df, figures_dir, label=""):
    os.makedirs(figures_dir, exist_ok=True)
    specs = [
        ("Accuracy", "Accuracy Comparison", "Accuracy", "accuracy_comparison.png", False),
        ("F1-score", "F1-score Comparison", "F1-score", "f1_comparison.png", False),
        ("Precision", "Precision Comparison", "Precision", "precision_comparison.png", False),
        ("Recall", "Recall Comparison", "Recall", "recall_comparison.png", False),
        ("Training Time (s)", "Training Time Comparison", "Seconds", "training_time_comparison.png", True),
        ("Single-Sample Latency (ms)", "Prediction Latency Comparison", "Milliseconds",
         "prediction_time_comparison.png", True),
        ("Serialized Model Size (KB)", "Model Size Comparison", "KB", "model_size_comparison.png", True),
        ("Deployment Friendliness (0-100)", "Deployment Friendliness Ranking", "Score",
         "deployment_friendliness.png", False),
    ]
    for col, title, ylabel, fname, asc in specs:
        bar_comparison(results_df, col, f"{title} — {label}", ylabel, os.path.join(figures_dir, fname), asc)

    radar_df = results_df.copy()
    radar_df["Speed (inv. latency)"] = -radar_df["Single-Sample Latency (ms)"]
    radar_df["Compactness (inv. size)"] = -radar_df["Serialized Model Size (KB)"]
    radar_metrics = ["Accuracy", "F1-score", "Speed (inv. latency)", "Compactness (inv. size)",
                      "Interpretability (1-5)"]
    radar_chart(radar_df, radar_metrics, os.path.join(figures_dir, "radar_chart.png"),
                f"Model Comparison Radar Chart — {label}")
    print(f"[{label}] All comparison visualizations saved to {figures_dir}/")


# ===========================================================================
# 8. PER-DATASET PIPELINE RUNNER (identical steps for UCI HAR and WISDM)
# ===========================================================================

def run_dataset_pipeline(dataset_name, data, args, out_root="."):
    print("\n" + "#" * 70)
    print(f"# DATASET: {dataset_name}")
    print("#" * 70)

    figures_dir = os.path.join(out_root, "figures", dataset_name)
    tables_dir = os.path.join(out_root, "tables", dataset_name)
    results_dir = os.path.join(out_root, "results", dataset_name)
    models_dir = os.path.join(out_root, "models", dataset_name)
    for d in [figures_dir, tables_dir, results_dir, models_dir,
              os.path.join(results_dir, "classification_reports"),
              os.path.join(results_dir, "saved_models")]:
        os.makedirs(d, exist_ok=True)

    verify_dimensions(data, label=dataset_name)
    run_eda(data, figures_dir, tables_dir, label=dataset_name)

    X_train_scaled, X_test_scaled, scaler = scale_features(
        data["X_train"], data["X_test"], save_path=os.path.join(models_dir, "scaler.joblib")
    )
    y_train_enc, y_test_enc, encoder = encode_labels(
        data["y_train_labels"], data["y_test_labels"], save_path=os.path.join(models_dir, "label_encoder.joblib")
    )
    integrity_check(data, label=dataset_name)
    label_names = list(encoder.classes_)

    models, param_grids = get_models_and_grids(fast_mode=args.fast)

    all_rows, param_table_rows = [], []

    for name, base_model in models.items():
        print(f"\n--- [{dataset_name}] {name} ---")

        if args.no_tune:
            best_estimator, best_params = base_model, base_model.get_params()
        else:
            grid = param_grids.get(name, {})
            if grid:
                search = GridSearchCV(base_model, grid, cv=args.cv, scoring="accuracy", n_jobs=-1)
                search.fit(X_train_scaled, y_train_enc)
                best_estimator, best_params = search.best_estimator_, search.best_params_
                print(f"Best params: {best_params}")
            else:
                best_estimator, best_params = base_model, base_model.get_params()

        clean_model = clone(best_estimator)
        fitted_model, tiny_metrics = evaluate_tinyml_metrics(
            name, clean_model, X_train_scaled, y_train_enc, X_test_scaled, models_dir=models_dir
        )

        perf_metrics, y_pred = evaluate_classifier(fitted_model, X_test_scaled, y_test_enc)
        save_classification_report(name, y_test_enc, y_pred, label_names,
                                    os.path.join(results_dir, "classification_reports"))
        plot_confusion_matrix(name, y_test_enc, y_pred, label_names, figures_dir)
        cv_metrics = run_cross_validation(fitted_model, X_train_scaled, y_train_enc, cv=args.cv)

        row = {"Model": name, **perf_metrics, **cv_metrics, **tiny_metrics}
        row["Model Complexity"] = json.dumps(row["Model Complexity"])
        all_rows.append(row)

        joblib.dump(fitted_model, os.path.join(results_dir, "saved_models", f"{name.replace(' ', '_')}.joblib"))
        param_table_rows.append({"Model": name, "Best Parameters": json.dumps(best_params, default=str)})

        print(f"Accuracy={perf_metrics['Accuracy']}  F1={perf_metrics['F1-score']}  "
              f"Size={tiny_metrics['Serialized Model Size (KB)']}KB  "
              f"Latency={tiny_metrics['Single-Sample Latency (ms)']}ms  "
              f"Deploy-Score={tiny_metrics['Deployment Friendliness (0-100)']}")

    results_df = pd.DataFrame(all_rows)
    pd.DataFrame(param_table_rows).to_csv(os.path.join(tables_dir, "table2_algorithm_parameters.csv"), index=False)
    results_df.to_csv(os.path.join(results_dir, "comparison_results.csv"), index=False)

    perf_cols = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "cv_mean_accuracy", "cv_std"]
    results_df[perf_cols].to_csv(os.path.join(tables_dir, "table3_performance_metrics.csv"), index=False)

    tiny_cols = ["Model", "Training Time (s)", "Single-Sample Latency (ms)", "Batch Throughput (samples/s)",
                 "Serialized Model Size (KB)", "Peak Memory - Python process (MiB)",
                 "Estimated Flash Usage (KB)", "Estimated RAM Usage (KB)",
                 "Interpretability (1-5)", "Deployment Friendliness (0-100)"]
    results_df[tiny_cols].to_csv(os.path.join(tables_dir, "table4_tinyml_metrics.csv"), index=False)

    ranking_df = results_df.copy()
    ranking_df["Composite Score"] = (
        0.6 * (ranking_df["F1-score"] * 100) + 0.4 * ranking_df["Deployment Friendliness (0-100)"]
    ).round(2)
    ranking_df = ranking_df.sort_values("Composite Score", ascending=False).reset_index(drop=True)
    ranking_df.insert(0, "Rank", ranking_df.index + 1)
    ranking_df[["Rank", "Model", "Accuracy", "F1-score", "Serialized Model Size (KB)",
                "Single-Sample Latency (ms)", "Deployment Friendliness (0-100)", "Composite Score"]].to_csv(
        os.path.join(tables_dir, "table5_overall_ranking.csv"), index=False
    )

    print(f"\n[{dataset_name}] RECOMMENDED MODEL: {ranking_df.iloc[0]['Model']}")
    print(ranking_df[["Rank", "Model", "Accuracy", "F1-score", "Deployment Friendliness (0-100)",
                       "Composite Score"]].to_string(index=False))

    generate_all_visualizations(results_df, figures_dir, label=dataset_name)

    return {"dataset_name": dataset_name, "results_df": results_df, "ranking_df": ranking_df}


# ===========================================================================
# 9. PHASE 2 — CROSS-DATASET COMPARISON
# ===========================================================================

def generate_comparison_tables(combined_df, tables_dir):
    os.makedirs(tables_dir, exist_ok=True)
    combined_df.to_csv(os.path.join(tables_dir, "table10_cross_dataset_comparison.csv"), index=False)

    wide_frames = []
    for col, _ in COMPARISON_METRICS:
        pivot = combined_df.pivot(index="Model", columns="Dataset", values=col)
        pivot.columns = [f"{ds} — {col}" for ds in pivot.columns]
        wide_frames.append(pivot)
    wide_df = pd.concat(wide_frames, axis=1).reset_index()
    wide_df.to_csv(os.path.join(tables_dir, "table10b_cross_dataset_wide.csv"), index=False)

    rank_rows = []
    for dataset in combined_df["Dataset"].unique():
        sub = combined_df[combined_df["Dataset"] == dataset].sort_values("Accuracy", ascending=False)
        rank_rows.append({"Dataset": dataset, "Top Model (by Accuracy)": sub.iloc[0]["Model"],
                           "Accuracy": sub.iloc[0]["Accuracy"]})
    pd.DataFrame(rank_rows).to_csv(os.path.join(tables_dir, "table10c_top_model_per_dataset.csv"), index=False)
    return wide_df


def generate_comparison_figures(combined_df, figures_dir):
    os.makedirs(figures_dir, exist_ok=True)
    datasets = combined_df["Dataset"].unique()

    for col, title in COMPARISON_METRICS:
        pivot = combined_df.pivot(index="Model", columns="Dataset", values=col)
        models = pivot.index.tolist()
        x = np.arange(len(models))
        width = 0.8 / max(len(datasets), 1)

        plt.figure(figsize=(10, 6))
        for i, dataset in enumerate(datasets):
            plt.bar(x + i * width, pivot[dataset], width=width, label=dataset)
        plt.xticks(x + width * (len(datasets) - 1) / 2, models, rotation=45, ha="right")
        plt.ylabel(title)
        plt.title(f"{title}: " + " vs. ".join(datasets))
        plt.legend()
        plt.tight_layout()
        fname = title.lower().replace(" ", "_").replace("(", "").replace(")", "")
        plt.savefig(os.path.join(figures_dir, f"cross_dataset_{fname}.png"), dpi=150)
        plt.close()

    print(f"Cross-dataset comparison figures saved to {figures_dir}/")


def compare_across_datasets(all_results, out_root="."):
    if len(all_results) < 2:
        print("\nOnly one dataset was evaluated — skipping Phase 2 cross-dataset comparison.")
        return

    print("\n" + "#" * 70)
    print("# PHASE 2: CROSS-DATASET COMPARISON (UCI HAR vs. WISDM)")
    print("#" * 70)

    merged_rows = []
    for res in all_results:
        for _, row in res["ranking_df"].iterrows():
            r = {"Dataset": res["dataset_name"], "Model": row["Model"]}
            for col, _ in COMPARISON_METRICS:
                r[col] = row[col]
            merged_rows.append(r)
    combined_df = pd.DataFrame(merged_rows)

    tables_dir = os.path.join(out_root, "tables", "cross_dataset")
    figures_dir = os.path.join(out_root, "figures", "cross_dataset")
    wide_df = generate_comparison_tables(combined_df, tables_dir)
    print(wide_df.to_string(index=False))

    generate_comparison_figures(combined_df, figures_dir)

    best_per_dataset = {res["dataset_name"]: res["ranking_df"].iloc[0]["Model"] for res in all_results}
    print("\nTop-ranked model per dataset:")
    for ds, m in best_per_dataset.items():
        print(f"  {ds}: {m}")
    if len(set(best_per_dataset.values())) == 1:
        winner = next(iter(best_per_dataset.values()))
        print(f"\n>> '{winner}' is the top-ranked model on EVERY evaluated dataset — "
              "early evidence the conclusion generalizes beyond a single dataset "
              "(formal statistical validation is Phase 5, not run here).")
    else:
        print("\n>> The top-ranked model differs across datasets — report both results "
              "transparently; see table10b_cross_dataset_wide.csv for the full breakdown.")

    print(f"\nCross-dataset comparison saved to {tables_dir}/ and {figures_dir}/")


# ===========================================================================
# 10. MAIN
# ===========================================================================

def parse_args():
    p = argparse.ArgumentParser(description="TinyML HAR pipeline — Phase 1 + Phase 2 (multi-dataset)")
    p.add_argument("--data-dir", default="data")
    p.add_argument("--datasets", default="uci,wisdm", help="Comma-separated: uci,wisdm")
    p.add_argument("--fast", action="store_true")
    p.add_argument("--no-tune", action="store_true")
    p.add_argument("--cv", type=int, default=5)
    p.add_argument("--no-download", action="store_true")
    p.add_argument("--wisdm-window", type=int, default=80)
    p.add_argument("--wisdm-step", type=int, default=40)
    # parse_known_args so this also behaves in Jupyter/Colab cells that inject
    # extra kernel args (e.g. -f /root/.local/share/.../kernel.json)
    args, _ = p.parse_known_args()
    return args


def main():
    args = parse_args()
    requested = [d.strip().lower() for d in args.datasets.split(",") if d.strip()]

    all_results = []

    if "uci" in requested:
        try:
            if not args.no_download:
                print("=" * 60)
                print("Ensuring UCI HAR dataset is available")
                print("=" * 60)
                download_uci_har(args.data_dir)
            data_uci = load_uci_har(args.data_dir)
            res = run_dataset_pipeline("UCI_HAR", data_uci, args)
            all_results.append(res)
        except Exception as e:
            print(f"\n[UCI HAR] Skipping dataset due to error: {e}")

    if "wisdm" in requested:
        try:
            if not args.no_download:
                print("\n" + "=" * 60)
                print("Ensuring WISDM dataset is available")
                print("=" * 60)
                download_wisdm(args.data_dir)
            data_wisdm = load_wisdm(args.data_dir, window_size=args.wisdm_window, step=args.wisdm_step)
            res = run_dataset_pipeline("WISDM", data_wisdm, args)
            all_results.append(res)
        except Exception as e:
            print(f"\n[WISDM] Skipping dataset due to error: {e}")

    if not all_results:
        print("\nNo datasets could be loaded. Nothing to do.")
        return

    compare_across_datasets(all_results)

    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE (Phase 1 + Phase 2)")
    print("=" * 70)
    print("See figures/<dataset>, tables/<dataset>, results/<dataset> for per-dataset outputs,")
    print("and tables/cross_dataset/, figures/cross_dataset/ for the Phase 2 comparison.")
    print("\nStopping here per scope — robustness testing (Phase 3), statistical validation")
    print("(Phase 5), and paper writing (Phase 6) are not included in this script.")

    return all_results


if __name__ == "__main__":
    main()

Ensuring UCI HAR dataset is available
[UCI HAR] Already present at 'data/UCI HAR Dataset', skipping download.

######################################################################
# DATASET: UCI_HAR
######################################################################
[UCI_HAR] X_train: (7352, 561)  y_train: (7352,)
[UCI_HAR] X_test : (2947, 561)  y_test : (2947,)
[UCI_HAR] Dimension check passed.
[UCI_HAR] Missing values in training set: 0
[UCI_HAR] EDA complete.
[UCI_HAR] Integrity checks: {'no_nan_train': True, 'no_nan_test': True, 'row_match_train': True, 'row_match_test': True, 'same_feature_count': True}

--- [UCI_HAR] Logistic Regression ---
Best params: {'C': 1}
Accuracy=0.9552  F1=0.9551  Size=27.21KB  Latency=0.2773ms  Deploy-Score=96.45

--- [UCI_HAR] Decision Tree ---
Best params: {'max_depth': 10, 'min_samples_split': 5}
Accuracy=0.8666  F1=0.866  Size=24.4KB  Latency=0.2415ms  Deploy-Score=96.84

--- [UCI_HAR] Random Forest ---
Best params: {'max_depth': None, 'n_estim

In [2]:
"""
TinyML HAR — Phase 3: Robustness Testing (Colab-ready, single file, STANDALONE)
====================================================================================
Towards TinyML Deployment: A Comparative Evaluation of Lightweight Machine
Learning Algorithms for Human Activity Recognition.

This is a SEPARATE, SELF-CONTAINED pipeline. It does not import, call, or
depend on the Phase 1 / Phase 2 scripts in any way, and writes to its own
output folder so nothing from earlier phases is touched or overwritten.

WHAT THIS SCRIPT DOES
----------------------------------------------------------------------------
1. Downloads UCI HAR (and optionally WISDM) and trains the same family of
   lightweight models used in Phase 1/2 (own copy of the training code —
   nothing is imported from those scripts).
2. Establishes each model's clean baseline accuracy/F1 on the untouched
   test set.
3. Re-evaluates every model under three corruption conditions, at several
   severity levels each:
      - Gaussian noise added to the (already standardized) features
      - Random feature removal (simulated sensor dropout — a random subset
        of features zeroed out per sample)
      - Sensor perturbation (multiplicative jitter per feature per sample,
        simulating calibration drift)
4. Produces:
      - a robustness results table (accuracy/F1 per model x condition x level)
      - a robustness summary table (average relative accuracy retention per
        model, across all corruption conditions — higher = more robust)
      - line-chart figures (accuracy vs. corruption severity, one line per
        model, one chart per condition)

HOW TO RUN IN GOOGLE COLAB
---------------------------
    !pip install -q memory_profiler xgboost seaborn
    !python har_tinyml_phase3_robustness.py

Datasets auto-download on first run into ./data (UCI HAR always; WISDM too
unless you pass --datasets uci).

OUTPUTS (kept entirely separate from Phase 1/2 folders)
----------------------------------------------------------------------------
phase3_robustness/tables/<dataset>/
    robustness_results.csv        - every (model, condition, level) data point
    robustness_summary.csv        - avg retention per model (higher = more robust)
phase3_robustness/figures/<dataset>/
    robustness_Gaussian_Noise.png
    robustness_Feature_Removal.png
    robustness_Sensor_Perturbation.png
phase3_robustness/results/<dataset>/
    baseline_results.csv          - clean accuracy/F1 per model, for reference

COMMAND-LINE FLAGS
----------------------------------------------------------------------------
    --data-dir DIR         Folder that holds/receives raw datasets (default: data)
    --datasets LIST         Comma-separated: "uci,wisdm" (default), "uci", "wisdm"
    --fast                  Tiny hyperparameter grids, quick smoke test
    --no-tune                Skip GridSearchCV, use default hyperparameters
    --cv N                   Cross-validation folds used during tuning (default 5)
    --no-download            Don't auto-download; expects data already present
    --wisdm-window N          WISDM window size in samples (default 80, ~4s @20Hz)
    --wisdm-step N             WISDM window stride (default 40, 50% overlap)
    --noise-levels LIST         Comma-separated Gaussian sigma values
                                  (default 0,0.05,0.1,0.2,0.3)
    --removal-fracs LIST         Comma-separated feature-removal fractions
                                   (default 0,0.1,0.2,0.3)
    --jitter-levels LIST          Comma-separated multiplicative jitter levels
                                    (default 0,0.05,0.1,0.2)
    --seed N                       Random seed for corruption sampling (default 42)
"""

import os
import io
import json
import time
import tarfile
import zipfile
import argparse
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

warnings.filterwarnings("ignore")

UCI_HAR_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]
WISDM_URLS = [
    "https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_ar_latest.tar.gz",
]

OUT_ROOT = "phase3_robustness"


# ===========================================================================
# 0. GENERIC ARCHIVE / SEARCH HELPERS
# ===========================================================================

def _find_dataset_root(search_dir):
    """Walk search_dir for the folder actually containing features.txt +
    train/ + test/ (UCI HAR's real root), regardless of enclosing folder name."""
    if not os.path.isdir(search_dir):
        return None
    for root, dirs, files in os.walk(search_dir):
        if "features.txt" in files and "train" in dirs and "test" in dirs:
            return root
    return None


def _find_file(search_dir, predicate):
    if not os.path.isdir(search_dir):
        return None
    for root, _, files in os.walk(search_dir):
        for fname in files:
            if predicate(fname):
                return os.path.join(root, fname)
    return None


def _is_wisdm_raw_file(fname):
    """Matches WISDM's actual data file ('WISDM_ar_v1.1_raw.txt') while
    excluding companion docs like 'WISDM_ar_v1.1_raw_about.txt' — the
    endswith("raw.txt") check is what makes that exclusion work, since the
    about file ends in "about.txt", not "raw.txt"."""
    f = fname.lower()
    return f.endswith("raw.txt") and "wisdm" in f


# ===========================================================================
# 1. UCI HAR — DOWNLOAD + LOAD
# ===========================================================================

def download_uci_har(data_dir="data"):
    target = os.path.join(data_dir, "UCI HAR Dataset")
    if os.path.isdir(target) and _find_dataset_root(target):
        print(f"[UCI HAR] Already present at '{target}', skipping download.")
        return

    os.makedirs(data_dir, exist_ok=True)
    import urllib.request
    import shutil

    last_err = None
    for url in UCI_HAR_URLS:
        try:
            print(f"[UCI HAR] Downloading from {url} ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                raw = resp.read()

            extract_dir = os.path.join(data_dir, "_uci_har_download_tmp")
            if os.path.isdir(extract_dir):
                shutil.rmtree(extract_dir)
            os.makedirs(extract_dir, exist_ok=True)

            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                zf.extractall(extract_dir)
            print("[UCI HAR] Top-level extraction complete.")

            found_root = _find_dataset_root(extract_dir)
            if not found_root:
                for root, _, files in os.walk(extract_dir):
                    for fname in files:
                        if fname.lower().endswith(".zip"):
                            nested_path = os.path.join(root, fname)
                            try:
                                with zipfile.ZipFile(nested_path) as nzf:
                                    nzf.extractall(root)
                                print(f"[UCI HAR] Extracted nested zip: {fname}")
                            except zipfile.BadZipFile:
                                continue
                found_root = _find_dataset_root(extract_dir)

            if not found_root:
                raise RuntimeError("Archive did not contain a recognizable UCI HAR structure.")

            if os.path.isdir(target):
                shutil.rmtree(target)
            shutil.move(found_root, target)
            shutil.rmtree(extract_dir, ignore_errors=True)
            print(f"[UCI HAR] Dataset ready at '{target}'.")
            return
        except Exception as e:
            last_err = e
            print(f"[UCI HAR]   Failed: {e}")
            continue

    raise RuntimeError(
        f"[UCI HAR] Could not auto-download (last error: {last_err}). Download manually from "
        "https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones "
        f"and unzip so that '{target}' exists, then re-run with --no-download."
    )


def _make_unique(names):
    seen = {}
    unique = []
    for n in names:
        if n in seen:
            seen[n] += 1
            unique.append(f"{n}_{seen[n]}")
        else:
            seen[n] = 0
            unique.append(n)
    return unique


def load_uci_har(data_dir="data"):
    dataset_root = os.path.join(data_dir, "UCI HAR Dataset")
    if not os.path.isdir(dataset_root) or not os.path.exists(os.path.join(dataset_root, "features.txt")):
        found = _find_dataset_root(data_dir)
        if found:
            dataset_root = found
        else:
            raise FileNotFoundError(
                f"Could not find a valid UCI HAR Dataset under '{data_dir}'. Run without "
                "--no-download to auto-fetch it."
            )

    features = pd.read_csv(os.path.join(dataset_root, "features.txt"), sep=r"\s+", header=None,
                            names=["idx", "name"])
    feature_names = _make_unique(features["name"].tolist())

    labels_df = pd.read_csv(os.path.join(dataset_root, "activity_labels.txt"), sep=r"\s+",
                             header=None, names=["id", "activity"])
    activity_map = dict(zip(labels_df["id"], labels_df["activity"]))

    def _load_split(split):
        split_dir = os.path.join(dataset_root, split)
        X = pd.read_csv(os.path.join(split_dir, f"X_{split}.txt"), sep=r"\s+", header=None)
        X.columns = feature_names
        y = pd.read_csv(os.path.join(split_dir, f"y_{split}.txt"), header=None, names=["Activity_Id"])
        return X, y["Activity_Id"].map(activity_map)

    X_train, y_train_labels = _load_split("train")
    X_test, y_test_labels = _load_split("test")

    return {
        "X_train": X_train.reset_index(drop=True), "X_test": X_test.reset_index(drop=True),
        "y_train_labels": y_train_labels.reset_index(drop=True),
        "y_test_labels": y_test_labels.reset_index(drop=True),
        "feature_names": feature_names,
    }


# ===========================================================================
# 2. WISDM — DOWNLOAD + WINDOWED-FEATURE EXTRACTION + LOAD
# ===========================================================================

def download_wisdm(data_dir="data"):
    existing = _find_file(data_dir, _is_wisdm_raw_file)
    if existing:
        print(f"[WISDM] Already present at '{existing}', skipping download.")
        return existing

    os.makedirs(data_dir, exist_ok=True)
    import urllib.request

    last_err = None
    for url in WISDM_URLS:
        try:
            print(f"[WISDM] Downloading from {url} ...")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=90) as resp:
                raw = resp.read()

            extract_dir = os.path.join(data_dir, "WISDM_ar_v1.1")
            os.makedirs(extract_dir, exist_ok=True)
            with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tf:
                tf.extractall(extract_dir)
            print("[WISDM] Extraction complete.")

            raw_file = _find_file(extract_dir, _is_wisdm_raw_file)
            if not raw_file:
                raise RuntimeError("Could not locate the raw .txt file inside the WISDM archive.")
            print(f"[WISDM] Dataset ready at '{raw_file}'.")
            return raw_file
        except Exception as e:
            last_err = e
            print(f"[WISDM]   Failed: {e}")
            continue

    print(
        f"[WISDM] WARNING: could not auto-download (last error: {last_err}). "
        "Continuing without WISDM — download manually from "
        "https://www.cis.fordham.edu/wisdm/dataset.php if you want WISDM robustness results."
    )
    return None


def parse_wisdm_raw(path):
    with open(path, "r", errors="ignore") as f:
        content = f.read()
    content = content.replace("\n", "").replace("\r", "")
    records = content.split(";")

    rows = []
    for rec in records:
        rec = rec.strip().strip(",")
        if not rec:
            continue
        parts = rec.split(",")
        if len(parts) < 6:
            continue
        try:
            user = int(parts[0])
            activity = parts[1].strip()
            x, y, z = float(parts[3]), float(parts[4]), float(parts[5])
        except ValueError:
            continue
        rows.append((user, activity, x, y, z))

    return pd.DataFrame(rows, columns=["user", "activity", "x", "y", "z"])


def _make_windows(df, window_size, step):
    df = df.reset_index(drop=True)
    change = (df["user"] != df["user"].shift()) | (df["activity"] != df["activity"].shift())
    run_id = change.cumsum()
    windows = []
    for _, group in df.groupby(run_id):
        group = group.reset_index(drop=True)
        n = len(group)
        for start in range(0, max(n - window_size + 1, 0), step):
            windows.append(group.iloc[start:start + window_size])
    return windows


def _safe_corr(a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def _extract_window_features(seg):
    feats = {}
    for axis in ("x", "y", "z"):
        vals = seg[axis].values.astype(float)
        feats[f"{axis}_mean"] = float(np.mean(vals))
        feats[f"{axis}_std"] = float(np.std(vals))
        feats[f"{axis}_min"] = float(np.min(vals))
        feats[f"{axis}_max"] = float(np.max(vals))
        feats[f"{axis}_range"] = feats[f"{axis}_max"] - feats[f"{axis}_min"]
        feats[f"{axis}_energy"] = float(np.mean(vals ** 2))
        feats[f"{axis}_mad"] = float(np.mean(np.abs(vals - np.mean(vals))))
        feats[f"{axis}_iqr"] = float(np.percentile(vals, 75) - np.percentile(vals, 25))

    x, y, z = seg["x"].values.astype(float), seg["y"].values.astype(float), seg["z"].values.astype(float)
    feats["corr_xy"] = _safe_corr(x, y)
    feats["corr_xz"] = _safe_corr(x, z)
    feats["corr_yz"] = _safe_corr(y, z)

    mag = np.sqrt(x ** 2 + y ** 2 + z ** 2)
    feats["mag_mean"] = float(np.mean(mag))
    feats["mag_std"] = float(np.std(mag))
    feats["mag_max"] = float(np.max(mag))
    feats["mag_min"] = float(np.min(mag))

    feats["activity"] = seg["activity"].iloc[0]
    feats["user"] = seg["user"].iloc[0]
    return feats


def extract_wisdm_features(raw_df, window_size=80, step=40):
    windows = _make_windows(raw_df, window_size, step)
    rows = [_extract_window_features(w) for w in windows if len(w) == window_size]
    return pd.DataFrame(rows)


def load_wisdm(data_dir="data", window_size=80, step=40, test_size=0.2, random_state=42):
    raw_path = _find_file(data_dir, _is_wisdm_raw_file)
    if not raw_path:
        raise FileNotFoundError(
            f"Could not find a WISDM raw .txt file under '{data_dir}'. Run without "
            "--no-download to auto-fetch it, or download manually from "
            "https://www.cis.fordham.edu/wisdm/dataset.php"
        )

    print(f"[WISDM] Parsing raw accelerometer log: {raw_path}")
    raw_df = parse_wisdm_raw(raw_path)
    if len(raw_df) == 0:
        raise ValueError(f"Parsed 0 readings from '{raw_path}' — wrong file may have been picked up.")
    print(f"[WISDM] Parsed {len(raw_df)} raw readings across {raw_df['user'].nunique()} users, "
          f"{raw_df['activity'].nunique()} activities.")

    print(f"[WISDM] Extracting sliding-window statistical features "
          f"(window={window_size} samples, step={step})...")
    feat_df = extract_wisdm_features(raw_df, window_size=window_size, step=step)
    if len(feat_df) == 0:
        raise ValueError("No sliding windows could be extracted — check window/step sizes.")
    print(f"[WISDM] Built {len(feat_df)} feature windows, {feat_df.shape[1] - 2} features.")

    X = feat_df.drop(columns=["activity", "user"])
    y_labels = feat_df["activity"]

    X_train, X_test, y_train_labels, y_test_labels = train_test_split(
        X, y_labels, test_size=test_size, random_state=random_state, stratify=y_labels
    )

    return {
        "X_train": X_train.reset_index(drop=True), "X_test": X_test.reset_index(drop=True),
        "y_train_labels": y_train_labels.reset_index(drop=True),
        "y_test_labels": y_test_labels.reset_index(drop=True),
        "feature_names": list(X.columns),
    }


# ===========================================================================
# 3. PREPROCESSING
# ===========================================================================

def scale_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


def encode_labels(y_train, y_test):
    encoder = LabelEncoder()
    y_train_enc = encoder.fit_transform(y_train)
    y_test_enc = encoder.transform(y_test)
    return y_train_enc, y_test_enc, encoder


# ===========================================================================
# 4. MODELS + HYPERPARAMETER GRIDS (same family as Phase 1/2, own copy)
# ===========================================================================

def get_models_and_grids(fast_mode=False):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
        "KNN": KNeighborsClassifier(),
        "Gaussian Naive Bayes": GaussianNB(),
        "Linear SVM": LinearSVC(max_iter=5000, random_state=42),
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBClassifier(eval_metric="mlogloss", random_state=42, n_jobs=-1)

    if fast_mode:
        param_grids = {
            "Logistic Regression": {"C": [1.0]},
            "Decision Tree": {"max_depth": [10]},
            "Random Forest": {"n_estimators": [50]},
            "KNN": {"n_neighbors": [5]},
            "Gaussian Naive Bayes": {},
            "Linear SVM": {"C": [1.0]},
            "XGBoost": {"n_estimators": [50], "max_depth": [4]},
        }
    else:
        param_grids = {
            "Logistic Regression": {"C": [0.01, 0.1, 1, 10]},
            "Decision Tree": {"max_depth": [5, 10, 20, None], "min_samples_split": [2, 5, 10]},
            "Random Forest": {"n_estimators": [50, 100, 200], "max_depth": [10, 20, None]},
            "KNN": {"n_neighbors": [3, 5, 7, 9], "weights": ["uniform", "distance"]},
            "Gaussian Naive Bayes": {"var_smoothing": [1e-9, 1e-8, 1e-7]},
            "Linear SVM": {"C": [0.01, 0.1, 1, 10]},
            "XGBoost": {"n_estimators": [100, 200], "max_depth": [3, 5, 7], "learning_rate": [0.05, 0.1]},
        }

    param_grids = {k: v for k, v in param_grids.items() if k in models}
    return models, param_grids


def train_all_models(X_train, y_train, args):
    """Trains every model (with optional GridSearchCV tuning) and returns a
    dict of fitted models. This is the ONLY job of this function — Phase 3
    doesn't need TinyML deployment metrics or the full Phase 1/2 table set,
    just fitted, representative models to stress-test."""
    models, param_grids = get_models_and_grids(fast_mode=args.fast)
    fitted = {}
    for name, base_model in models.items():
        print(f"  Training {name}...")
        if args.no_tune:
            best_estimator = base_model
        else:
            grid = param_grids.get(name, {})
            if grid:
                search = GridSearchCV(base_model, grid, cv=args.cv, scoring="accuracy", n_jobs=-1)
                search.fit(X_train, y_train)
                best_estimator = search.best_estimator_
            else:
                best_estimator = base_model
        model = clone(best_estimator)
        model.fit(X_train, y_train)
        fitted[name] = model
    return fitted


# ===========================================================================
# 5. BASELINE EVALUATION
# ===========================================================================

def evaluate_baseline(fitted_models, X_test, y_test):
    rows = []
    for name, model in fitted_models.items():
        pred = model.predict(X_test)
        rows.append({
            "Model": name,
            "Accuracy": round(accuracy_score(y_test, pred), 4),
            "Precision": round(precision_score(y_test, pred, average="weighted", zero_division=0), 4),
            "Recall": round(recall_score(y_test, pred, average="weighted", zero_division=0), 4),
            "F1-score": round(f1_score(y_test, pred, average="weighted", zero_division=0), 4),
        })
    return pd.DataFrame(rows)


# ===========================================================================
# 6. PHASE 3 — ROBUSTNESS TESTING (the core of this script)
# ===========================================================================

def add_gaussian_noise(X, sigma, rng):
    """Adds N(0, sigma^2) noise to every (already-standardized) feature —
    sigma is directly comparable across features since they're all on a
    z-score scale. Simulates sensor noise / signal jitter."""
    return X + rng.normal(0, sigma, size=X.shape) if sigma > 0 else X


def remove_random_features(X, frac, rng):
    """Zeroes out a random `frac` fraction of features, independently per
    sample — simulates a dropped/failed sensor channel for that reading."""
    if frac <= 0:
        return X
    Xc = X.copy()
    n_samples, n_features = Xc.shape
    n_remove = max(int(round(frac * n_features)), 1)
    for i in range(n_samples):
        idx = rng.choice(n_features, size=n_remove, replace=False)
        Xc[i, idx] = 0.0
    return Xc


def perturb_sensor_readings(X, jitter, rng):
    """Multiplies every feature by (1 + U(-jitter, jitter)), independently
    per sample per feature — simulates sensor calibration drift."""
    if jitter <= 0:
        return X
    factors = 1 + rng.uniform(-jitter, jitter, size=X.shape)
    return X * factors


def evaluate_robustness(model_name, model, X_test, y_test, seed,
                         noise_levels, removal_fracs, jitter_levels):
    rng = np.random.RandomState(seed)
    baseline_acc = accuracy_score(y_test, model.predict(X_test))
    rows = []

    def _row(condition, level, Xp):
        pred = model.predict(Xp)
        acc = accuracy_score(y_test, pred)
        f1 = f1_score(y_test, pred, average="weighted", zero_division=0)
        retention = round(acc / baseline_acc, 4) if baseline_acc > 0 else np.nan
        return {"Model": model_name, "Condition": condition, "Level": level,
                "Accuracy": round(acc, 4), "F1-score": round(f1, 4),
                "Relative Retention": retention}

    for level in noise_levels:
        rows.append(_row("Gaussian Noise", level, add_gaussian_noise(X_test, level, rng)))
    for frac in removal_fracs:
        rows.append(_row("Feature Removal", frac, remove_random_features(X_test, frac, rng)))
    for level in jitter_levels:
        rows.append(_row("Sensor Perturbation", level, perturb_sensor_readings(X_test, level, rng)))

    return pd.DataFrame(rows)


def plot_robustness_curves(robustness_all_df, figures_dir, label=""):
    os.makedirs(figures_dir, exist_ok=True)
    for condition in robustness_all_df["Condition"].unique():
        sub = robustness_all_df[robustness_all_df["Condition"] == condition]
        plt.figure(figsize=(9, 5))
        for model_name in sub["Model"].unique():
            s = sub[sub["Model"] == model_name].sort_values("Level")
            plt.plot(s["Level"], s["Accuracy"], marker="o", label=model_name)
        plt.xlabel("Corruption Level")
        plt.ylabel("Accuracy")
        plt.title(f"Robustness under {condition} — {label}")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(os.path.join(figures_dir, f"robustness_{condition.replace(' ', '_')}.png"), dpi=150)
        plt.close()
    print(f"[{label}] Robustness curve figures saved to {figures_dir}/")


# ===========================================================================
# 7. PER-DATASET RUNNER
# ===========================================================================

def run_robustness_pipeline(dataset_name, data, args):
    print("\n" + "#" * 70)
    print(f"# PHASE 3 ROBUSTNESS TESTING — DATASET: {dataset_name}")
    print("#" * 70)

    tables_dir = os.path.join(OUT_ROOT, "tables", dataset_name)
    figures_dir = os.path.join(OUT_ROOT, "figures", dataset_name)
    results_dir = os.path.join(OUT_ROOT, "results", dataset_name)
    for d in [tables_dir, figures_dir, results_dir]:
        os.makedirs(d, exist_ok=True)

    print(f"[{dataset_name}] X_train: {data['X_train'].shape}   X_test: {data['X_test'].shape}")

    X_train_scaled, X_test_scaled, scaler = scale_features(data["X_train"], data["X_test"])
    y_train_enc, y_test_enc, encoder = encode_labels(data["y_train_labels"], data["y_test_labels"])

    print(f"\n[{dataset_name}] Training models...")
    fitted_models = train_all_models(X_train_scaled, y_train_enc, args)

    baseline_df = evaluate_baseline(fitted_models, X_test_scaled, y_test_enc)
    baseline_df.to_csv(os.path.join(results_dir, "baseline_results.csv"), index=False)
    print(f"\n[{dataset_name}] Baseline (clean) performance:")
    print(baseline_df.to_string(index=False))

    print(f"\n[{dataset_name}] Running robustness tests "
          f"(Gaussian noise, feature removal, sensor perturbation)...")
    robustness_parts = []
    for name, model in fitted_models.items():
        robustness_parts.append(evaluate_robustness(
            name, model, X_test_scaled, y_test_enc, seed=args.seed,
            noise_levels=args.noise_levels, removal_fracs=args.removal_fracs,
            jitter_levels=args.jitter_levels,
        ))
    robustness_df = pd.concat(robustness_parts, ignore_index=True)
    robustness_df.to_csv(os.path.join(tables_dir, "robustness_results.csv"), index=False)

    summary_df = (robustness_df[robustness_df["Level"] > 0]
                  .groupby("Model")["Relative Retention"].mean().reset_index()
                  .rename(columns={"Relative Retention": "Avg Retention Under Corruption"})
                  .sort_values("Avg Retention Under Corruption", ascending=False))
    summary_df.to_csv(os.path.join(tables_dir, "robustness_summary.csv"), index=False)

    print(f"\n[{dataset_name}] Robustness summary (higher = more robust to corruption):")
    print(summary_df.to_string(index=False))

    plot_robustness_curves(robustness_df, figures_dir, label=dataset_name)

    print(f"\n[{dataset_name}] Outputs saved under {OUT_ROOT}/(tables|figures|results)/{dataset_name}/")
    return {"dataset_name": dataset_name, "baseline_df": baseline_df,
            "robustness_df": robustness_df, "summary_df": summary_df}


# ===========================================================================
# 8. MAIN
# ===========================================================================

def _parse_float_list(s):
    return [float(x) for x in s.split(",") if x.strip() != ""]


def parse_args():
    p = argparse.ArgumentParser(description="TinyML HAR — Phase 3 robustness testing (standalone)")
    p.add_argument("--data-dir", default="data")
    p.add_argument("--datasets", default="uci,wisdm", help="Comma-separated: uci,wisdm")
    p.add_argument("--fast", action="store_true")
    p.add_argument("--no-tune", action="store_true")
    p.add_argument("--cv", type=int, default=5)
    p.add_argument("--no-download", action="store_true")
    p.add_argument("--wisdm-window", type=int, default=80)
    p.add_argument("--wisdm-step", type=int, default=40)
    p.add_argument("--noise-levels", type=str, default="0,0.05,0.1,0.2,0.3")
    p.add_argument("--removal-fracs", type=str, default="0,0.1,0.2,0.3")
    p.add_argument("--jitter-levels", type=str, default="0,0.05,0.1,0.2")
    p.add_argument("--seed", type=int, default=42)
    args, _ = p.parse_known_args()

    args.noise_levels = _parse_float_list(args.noise_levels)
    args.removal_fracs = _parse_float_list(args.removal_fracs)
    args.jitter_levels = _parse_float_list(args.jitter_levels)
    return args


def main():
    args = parse_args()
    requested = [d.strip().lower() for d in args.datasets.split(",") if d.strip()]

    all_results = []

    if "uci" in requested:
        try:
            if not args.no_download:
                print("=" * 60)
                print("Ensuring UCI HAR dataset is available")
                print("=" * 60)
                download_uci_har(args.data_dir)
            data_uci = load_uci_har(args.data_dir)
            res = run_robustness_pipeline("UCI_HAR", data_uci, args)
            all_results.append(res)
        except Exception as e:
            print(f"\n[UCI HAR] Skipping dataset due to error: {e}")

    if "wisdm" in requested:
        try:
            if not args.no_download:
                print("\n" + "=" * 60)
                print("Ensuring WISDM dataset is available")
                print("=" * 60)
                download_wisdm(args.data_dir)
            data_wisdm = load_wisdm(args.data_dir, window_size=args.wisdm_window, step=args.wisdm_step)
            res = run_robustness_pipeline("WISDM", data_wisdm, args)
            all_results.append(res)
        except Exception as e:
            print(f"\n[WISDM] Skipping dataset due to error: {e}")

    if not all_results:
        print("\nNo datasets could be loaded. Nothing to do.")
        return

    print("\n" + "=" * 70)
    print("PHASE 3 ROBUSTNESS TESTING COMPLETE")
    print("=" * 70)
    for res in all_results:
        most_robust = res["summary_df"].iloc[0]
        least_robust = res["summary_df"].iloc[-1]
        print(f"[{res['dataset_name']}] Most robust: {most_robust['Model']} "
              f"({most_robust['Avg Retention Under Corruption']:.3f} avg retention)  |  "
              f"Least robust: {least_robust['Model']} "
              f"({least_robust['Avg Retention Under Corruption']:.3f} avg retention)")
    print(f"\nAll outputs under ./{OUT_ROOT}/ — this is a separate folder from any "
          "Phase 1/2 outputs, nothing was merged or overwritten.")

    return all_results


if __name__ == "__main__":
    main()

Ensuring UCI HAR dataset is available
[UCI HAR] Downloading from https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip ...
[UCI HAR] Top-level extraction complete.
[UCI HAR] Extracted nested zip: UCI HAR Dataset.zip
[UCI HAR] Dataset ready at 'data/UCI HAR Dataset'.

######################################################################
# PHASE 3 ROBUSTNESS TESTING — DATASET: UCI_HAR
######################################################################
[UCI_HAR] X_train: (7352, 561)   X_test: (2947, 561)

[UCI_HAR] Training models...
  Training Logistic Regression...
  Training Decision Tree...
  Training Random Forest...
  Training KNN...
  Training Gaussian Naive Bayes...
  Training Linear SVM...
  Training XGBoost...

[UCI_HAR] Baseline (clean) performance:
               Model  Accuracy  Precision  Recall  F1-score
 Logistic Regression    0.9552     0.9569  0.9552    0.9551
       Decision Tree    0.8666     0.8678  0.8666    0.8660
       

In [6]:
"""
TinyML HAR — Phase 4: TinyML-Oriented Analysis (Colab-ready, single file, STANDALONE)
==========================================================================================
Towards TinyML Deployment: A Comparative Evaluation of Lightweight Machine
Learning Algorithms for Human Activity Recognition.

THIS SCRIPT DOES NOT LOAD ANY DATASET AND DOES NOT TRAIN ANY MODEL.
----------------------------------------------------------------------------
Every earlier pipeline script (main.py / har_tinyml.py / har_tinyml_phase2.py)
already computed and saved, per model, per dataset:

    Accuracy, F1-score, Training Time, Prediction Latency,
    Serialized Model Size, Estimated RAM, Estimated Flash,
    Deployment Friendliness score

into a `comparison_results.csv` file. This script just FINDS those files,
reads them, builds the formal "TinyML-Oriented Analysis" table (the 7
metrics above + a Deployment Suitability verdict), and produces the
comparison figures — nothing is retrained or re-downloaded.

WHERE IT LOOKS FOR RESULTS
----------------------------------------------------------------------------
It auto-detects whichever of these layouts you already have on disk:
    results/comparison_results.csv                  (flat — single dataset,
                                                       e.g. from main.py or
                                                       the Phase 1 single-file
                                                       script)
    results/<DATASET_NAME>/comparison_results.csv    (nested — one folder per
                                                        dataset, e.g. from
                                                        main_wisdm.py or the
                                                        Phase 1+2 single-file
                                                        script)
Any number of datasets found (1, 2, or more) will be analyzed; if 2+ are
found it also produces a cross-dataset overlay of every metric.

HOW TO RUN IN GOOGLE COLAB
---------------------------
    !python har_tinyml_phase4_analysis.py

(no pip installs needed beyond what you already have — this script only
uses pandas/numpy/matplotlib, no sklearn/xgboost/memory_profiler required
since nothing is trained here.)

If your results live somewhere other than ./results, point at it with
`--results-dir` (see flags below).

OUTPUTS (kept entirely separate from every earlier phase's folders)
----------------------------------------------------------------------------
phase4_tinyml_analysis/tables/<dataset>/
    tinyml_analysis_table.csv       - the 7 requested metrics + suitability verdict
phase4_tinyml_analysis/figures/<dataset>/
    accuracy_comparison.png, f1_comparison.png, model_size_comparison.png,
    prediction_latency_comparison.png, training_time_comparison.png,
    estimated_ram_comparison.png, estimated_flash_comparison.png,
    deployment_friendliness_comparison.png,
    deployment_landscape.png        - Flash vs RAM scatter, bubble=accuracy,
                                       color=deployment tier (the signature
                                       "does this fit on an MCU" chart)
phase4_tinyml_analysis/tables/cross_dataset_tinyml_analysis.csv   (if 2+ datasets)
phase4_tinyml_analysis/figures/cross_dataset/<metric>.png          (if 2+ datasets)

COMMAND-LINE FLAGS
----------------------------------------------------------------------------
    --results-dir DIR   Root folder to search for comparison_results.csv
                          files (default: results)
    --flash-tiers        Override the Flash/RAM tier thresholds (KB), as
                           "name:flash:ram,name:flash:ram,..."
                           default matches common MCU classes (see below)
    --latency-ok-ms N     Latency (ms) considered "fine for real-time"
                            (default 50)
    --latency-warn-ms N    Latency (ms) above which it's flagged "high
                             latency" rather than just "borderline" (default 200)
"""

import os
import argparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_ROOT = "phase4_tinyml_analysis"

# Default MCU deployment tiers: (name, max Flash KB, max RAM KB). A model is
# placed in the SMALLEST tier whose budget it fits within. These are rough,
# commonly-cited ballparks — not a substitute for validating on real
# hardware after cross-compilation (TFLite Micro / emlearn / m2cgen).
DEFAULT_TIERS = [
    ("Class 0 — Ultra-constrained (8-bit MCU, e.g. ATtiny)", 32, 2),
    ("Class 1 — Entry Cortex-M0/M0+", 256, 32),
    ("Class 2 — Mainstream Cortex-M4 / ESP32", 1024, 256),
]
FALLBACK_TIER = ("Not TinyML-suitable as-is (exceeds typical MCU budgets — "
                  "needs pruning/quantization, or target an MPU-class device)")

# The Phase 4 metric set, mapped from comparison_results.csv's column names
# to clean, paper-ready labels.
METRIC_MAP = [
    ("Accuracy", "Accuracy", False),
    ("F1-score", "F1-score", False),
    ("Serialized Model Size (KB)", "Model Size (KB)", True),
    ("Single-Sample Latency (ms)", "Prediction Latency (ms)", True),
    ("Training Time (s)", "Training Time (s)", True),
    ("Estimated RAM Usage (KB)", "Estimated RAM (KB)", True),
    ("Estimated Flash Usage (KB)", "Estimated Flash (KB)", True),
    ("Deployment Friendliness (0-100)", "Deployment Friendliness (0-100)", False),
]
REQUIRED_SOURCE_COLS = ["Model"] + [src for src, _, _ in METRIC_MAP]


# ===========================================================================
# 1. FIND + LOAD ALREADY-COMPUTED RESULTS
# ===========================================================================

def find_result_csvs(results_dir="results"):
    """Auto-detects flat (single dataset) or nested (per-dataset subfolder)
    comparison_results.csv layouts, from any of the earlier pipeline scripts."""
    found = {}

    flat_path = os.path.join(results_dir, "comparison_results.csv")
    if os.path.exists(flat_path):
        found["Dataset"] = flat_path

    if os.path.isdir(results_dir):
        for entry in sorted(os.listdir(results_dir)):
            sub = os.path.join(results_dir, entry)
            csv_path = os.path.join(sub, "comparison_results.csv")
            if os.path.isdir(sub) and os.path.exists(csv_path):
                found[entry] = csv_path

    return found


def load_and_validate(csv_path, label):
    df = pd.read_csv(csv_path)
    missing = [c for c in REQUIRED_SOURCE_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"[{label}] '{csv_path}' is missing expected column(s): {missing}. "
            "This doesn't look like a comparison_results.csv produced by the "
            "earlier pipeline scripts."
        )
    return df


# ===========================================================================
# 2. DEPLOYMENT SUITABILITY CLASSIFICATION
# ===========================================================================

def parse_tiers_arg(s):
    if not s:
        return DEFAULT_TIERS
    tiers = []
    for part in s.split(","):
        name, flash, ram = part.split(":")
        tiers.append((name.strip(), float(flash), float(ram)))
    return tiers


def classify_deployment(flash_kb, ram_kb, latency_ms, tiers, latency_ok_ms, latency_warn_ms):
    verdict = FALLBACK_TIER
    for tier_name, flash_limit, ram_limit in tiers:
        if flash_kb <= flash_limit and ram_kb <= ram_limit:
            verdict = tier_name
            break

    if latency_ms <= latency_ok_ms:
        latency_note = "OK for real-time"
    elif latency_ms <= latency_warn_ms:
        latency_note = "Borderline — consider optimization"
    else:
        latency_note = "High latency — likely needs optimization"

    return verdict, latency_note


# ===========================================================================
# 3. BUILD THE TINYML-ORIENTED ANALYSIS TABLE
# ===========================================================================

def build_tinyml_table(df, tiers, latency_ok_ms, latency_warn_ms):
    rows = []
    for _, r in df.iterrows():
        verdict, latency_note = classify_deployment(
            r["Estimated Flash Usage (KB)"], r["Estimated RAM Usage (KB)"],
            r["Single-Sample Latency (ms)"], tiers, latency_ok_ms, latency_warn_ms,
        )
        row = {"Model": r["Model"]}
        for src, label, _ in METRIC_MAP:
            row[label] = r[src]
        row["Deployment Tier"] = verdict
        row["Latency Note"] = latency_note
        rows.append(row)

    table = pd.DataFrame(rows).sort_values("Deployment Friendliness (0-100)", ascending=False).reset_index(drop=True)
    table.insert(0, "Rank", table.index + 1)
    return table


# ===========================================================================
# 4. FIGURES
# ===========================================================================

def bar_comparison(df, metric_col, title, ylabel, out_path, ascending=False):
    plt.figure(figsize=(9, 5))
    sorted_df = df.sort_values(metric_col, ascending=ascending)
    bars = plt.bar(sorted_df["Model"], sorted_df[metric_col], color="#4C72B0")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, h, f"{h:.3g}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_per_metric_bars(table, figures_dir, label):
    os.makedirs(figures_dir, exist_ok=True)
    specs = [
        ("Accuracy", "Accuracy Comparison", "Accuracy", "accuracy_comparison.png", False),
        ("F1-score", "F1-score Comparison", "F1-score", "f1_comparison.png", False),
        ("Model Size (KB)", "Model Size Comparison", "KB", "model_size_comparison.png", True),
        ("Prediction Latency (ms)", "Prediction Latency Comparison", "Milliseconds",
         "prediction_latency_comparison.png", True),
        ("Training Time (s)", "Training Time Comparison", "Seconds", "training_time_comparison.png", True),
        ("Estimated RAM (KB)", "Estimated RAM Usage Comparison", "KB", "estimated_ram_comparison.png", True),
        ("Estimated Flash (KB)", "Estimated Flash Usage Comparison", "KB", "estimated_flash_comparison.png", True),
        ("Deployment Friendliness (0-100)", "Deployment Friendliness Ranking", "Score",
         "deployment_friendliness_comparison.png", False),
    ]
    for col, title, ylabel, fname, asc in specs:
        bar_comparison(table, col, f"{title} — {label}", ylabel, os.path.join(figures_dir, fname), asc)
    print(f"[{label}] Per-metric comparison figures saved to {figures_dir}/")


def plot_deployment_landscape(table, out_path, label):
    """The signature TinyML chart: Estimated Flash vs. Estimated RAM
    (log-log), bubble size = accuracy, color = deployment tier. Lets you see
    at a glance which models actually fit a given MCU budget."""
    tiers_present = table["Deployment Tier"].unique()
    cmap = plt.colormaps.get_cmap("tab10")
    tier_color = {t: cmap(i) for i, t in enumerate(tiers_present)}

    plt.figure(figsize=(9, 7))
    for _, row in table.iterrows():
        plt.scatter(
            row["Estimated Flash (KB)"], row["Estimated RAM (KB)"],
            s=max(row["Accuracy"], 0.05) * 400,
            color=tier_color[row["Deployment Tier"]], edgecolor="black", alpha=0.85, zorder=3,
        )
        plt.annotate(row["Model"], (row["Estimated Flash (KB)"], row["Estimated RAM (KB)"]),
                     fontsize=8, xytext=(6, 6), textcoords="offset points")

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Estimated Flash Usage (KB, log scale)")
    plt.ylabel("Estimated RAM Usage (KB, log scale)")
    plt.title(f"TinyML Deployment Landscape — {label}\n(bubble size = accuracy, color = deployment tier)")

    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=tier_color[t],
                           markersize=10, label=t) for t in tiers_present]
    plt.legend(handles=handles, fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()


def plot_cross_dataset_metric(combined_df, col, title, figures_dir):
    pivot = combined_df.pivot(index="Model", columns="Dataset", values=col)
    models = pivot.index.tolist()
    datasets = pivot.columns.tolist()
    x = np.arange(len(models))
    width = 0.8 / max(len(datasets), 1)

    plt.figure(figsize=(10, 6))
    for i, dataset in enumerate(datasets):
        plt.bar(x + i * width, pivot[dataset], width=width, label=dataset)
    plt.xticks(x + width * (len(datasets) - 1) / 2, models, rotation=45, ha="right")
    plt.ylabel(title)
    plt.title(f"{title}: " + " vs. ".join(datasets))
    plt.legend()
    plt.tight_layout()
    fname = title.lower().replace(" ", "_").replace("(", "").replace(")", "")
    plt.savefig(os.path.join(figures_dir, f"{fname}.png"), dpi=150)
    plt.close()


# ===========================================================================
# 5. PER-DATASET RUNNER
# ===========================================================================

def run_analysis_for_dataset(label, csv_path, tiers, latency_ok_ms, latency_warn_ms):
    print(f"\n[{label}] Reading already-computed results from '{csv_path}'...")
    df = load_and_validate(csv_path, label)

    table = build_tinyml_table(df, tiers, latency_ok_ms, latency_warn_ms)

    tables_dir = os.path.join(OUT_ROOT, "tables", label)
    figures_dir = os.path.join(OUT_ROOT, "figures", label)
    os.makedirs(tables_dir, exist_ok=True)
    os.makedirs(figures_dir, exist_ok=True)

    table.to_csv(os.path.join(tables_dir, "tinyml_analysis_table.csv"), index=False)

    print(f"\n[{label}] TinyML-Oriented Analysis (ranked by Deployment Friendliness):")
    print(table[["Rank", "Model", "Accuracy", "F1-score", "Model Size (KB)",
                  "Prediction Latency (ms)", "Training Time (s)", "Estimated RAM (KB)",
                  "Estimated Flash (KB)", "Deployment Friendliness (0-100)",
                  "Deployment Tier"]].to_string(index=False))

    plot_per_metric_bars(table, figures_dir, label)
    plot_deployment_landscape(table, os.path.join(figures_dir, "deployment_landscape.png"), label)
    print(f"[{label}] Deployment landscape figure saved to {figures_dir}/deployment_landscape.png")

    top = table.iloc[0]
    print(f"\n[{label}] TinyML-recommended model: {top['Model']} "
          f"(Accuracy={top['Accuracy']}, F1={top['F1-score']}, "
          f"Size={top['Model Size (KB)']}KB, Tier: {top['Deployment Tier']})")

    return table


# ===========================================================================
# 6. MAIN
# ===========================================================================

def parse_args():
    p = argparse.ArgumentParser(
        description="TinyML HAR — Phase 4 TinyML-oriented analysis (standalone, no retraining)"
    )
    p.add_argument("--results-dir", default="results",
                    help="Root folder to search for comparison_results.csv files")
    p.add_argument("--flash-tiers", default="",
                    help='Override tiers as "name:flash_kb:ram_kb,name:flash_kb:ram_kb,..."')
    p.add_argument("--latency-ok-ms", type=float, default=50.0)
    p.add_argument("--latency-warn-ms", type=float, default=200.0)
    args, _ = p.parse_known_args()
    return args


def main():
    args = parse_args()
    tiers = parse_tiers_arg(args.flash_tiers)

    found = find_result_csvs(args.results_dir)
    if not found:
        print(
            f"No comparison_results.csv found under '{args.results_dir}'.\n"
            "This script reads results already produced by an earlier pipeline run — "
            "run main.py, har_tinyml.py, or har_tinyml_phase2.py first, or point at the "
            "right folder with --results-dir."
        )
        return

    print("=" * 70)
    print(f"PHASE 4: TinyML-Oriented Analysis — found {len(found)} dataset(s): "
          f"{', '.join(found.keys())}")
    print("(reading already-computed results — no data loading, no retraining)")
    print("=" * 70)

    all_tables = {}
    for label, csv_path in found.items():
        try:
            all_tables[label] = run_analysis_for_dataset(
                label, csv_path, tiers, args.latency_ok_ms, args.latency_warn_ms
            )
        except Exception as e:
            print(f"\n[{label}] Skipping — {e}")

    if len(all_tables) >= 2:
        print("\n" + "#" * 70)
        print("# CROSS-DATASET TinyML METRIC OVERLAY")
        print("#" * 70)
        combined_rows = []
        for label, table in all_tables.items():
            for _, row in table.iterrows():
                r = {"Dataset": label, "Model": row["Model"]}
                for _, dst_label, _ in METRIC_MAP:
                    r[dst_label] = row[dst_label]
                combined_rows.append(r)
        combined_df = pd.DataFrame(combined_rows)

        cross_tables_dir = os.path.join(OUT_ROOT, "tables")
        cross_figures_dir = os.path.join(OUT_ROOT, "figures", "cross_dataset")
        os.makedirs(cross_tables_dir, exist_ok=True)
        os.makedirs(cross_figures_dir, exist_ok=True)
        combined_df.to_csv(os.path.join(cross_tables_dir, "cross_dataset_tinyml_analysis.csv"), index=False)

        for _, dst_label, _ in METRIC_MAP:
            plot_cross_dataset_metric(combined_df, dst_label, dst_label, cross_figures_dir)

        print(f"Cross-dataset TinyML comparison saved to {cross_tables_dir}/cross_dataset_tinyml_analysis.csv "
              f"and {cross_figures_dir}/")

    print("\n" + "=" * 70)
    print("PHASE 4 COMPLETE")
    print("=" * 70)
    print(f"All outputs under ./{OUT_ROOT}/ — separate folder, nothing from earlier "
          "phases was touched, nothing was retrained.")


if __name__ == "__main__":
    main()

PHASE 4: TinyML-Oriented Analysis — found 2 dataset(s): UCI_HAR, WISDM
(reading already-computed results — no data loading, no retraining)

[UCI_HAR] Reading already-computed results from 'results/UCI_HAR/comparison_results.csv'...

[UCI_HAR] TinyML-Oriented Analysis (ranked by Deployment Friendliness):
 Rank                Model  Accuracy  F1-score  Model Size (KB)  Prediction Latency (ms)  Training Time (s)  Estimated RAM (KB)  Estimated Flash (KB)  Deployment Friendliness (0-100)                                                                                                     Deployment Tier
    1        Decision Tree    0.8666    0.8660            24.40                   0.2415             5.8907               12.20                 24.40                            96.84                                                                                       Class 1 — Entry Cortex-M0/M0+
    2  Logistic Regression    0.9552    0.9551            27.21                   0.2773         

In [7]:
"""
TinyML HAR — Phase 5: Statistical Validation (Colab-ready, single file, STANDALONE)
========================================================================================
Towards TinyML Deployment: A Comparative Evaluation of Lightweight Machine
Learning Algorithms for Human Activity Recognition.

THIS SCRIPT DOES NOT TRAIN (FIT) ANY MODEL.
----------------------------------------------------------------------------
It loads the models you already trained and saved (results/<dataset>/saved_models/
*.joblib) and runs them in inference-only mode (.predict()) against the same
held-out test set they were originally evaluated on.

WHY NOT JUST WILCOXON/FRIEDMAN ON CV FOLDS?
----------------------------------------------------------------------------
Classic paired Wilcoxon signed-rank and Friedman tests need multiple
independent paired scores — normally one score per cross-validation fold,
which requires refitting a model on each fold's training split. Since
retraining is explicitly off the table here, using those tests on anything
we can produce without refitting (e.g. bootstrap-resampling a single test
set and feeding it into Wilcoxon/Friedman as if the resamples were
independent folds) would be statistically questionable — bootstrap
resamples of the same test set are not independent, which inflates false
positives.

The methodologically correct substitute for "compare already-trained
classifiers on one held-out test set, no retraining" is well established
(Dietterich, 1998, "Approximate Statistical Tests for Comparing Supervised
Classification Learning Algorithms", Neural Computation):

    - McNemar's exact test        -> pairwise comparison of two classifiers'
                                       predictions on the same test set
                                       (plays the role Wilcoxon would play)
    - Cochran's Q test            -> omnibus test across ALL classifiers on
                                       the same test set (the single-test-set
                                       generalization of McNemar — plays the
                                       role Friedman would play)
    - Holm-Bonferroni corrected
      pairwise McNemar            -> post-hoc analysis, run only if Cochran's
                                       Q is significant (mirrors doing Nemenyi
                                       only after a significant Friedman test)

This script implements exactly that, plus two flavors of confidence
interval so the "5/10-fold CV" and "confidence intervals" asks are still
covered without retraining:

    - An approximate CI from the k-fold CV Phase 1/2 ALREADY ran during
      training (cv_mean_accuracy / cv_std, saved in comparison_results.csv),
      using a t-distribution across the k folds.
    - A Wilson score CI computed directly on test-set accuracy (the
      standard exact-ish CI for a binomial proportion — no resampling
      needed).

HOW TO RUN IN GOOGLE COLAB
---------------------------
    !python har_tinyml_phase5_statistics.py

Needs your dataset still available (or re-downloadable) under ./data, plus
whatever earlier script saved results/<dataset>/saved_models/*.joblib and
models/<dataset>/{scaler,label_encoder}.joblib — i.e. har_tinyml.py or
har_tinyml_phase2.py must have been run first. If those aren't found the
script tells you exactly what's missing rather than trying to train
anything.

OUTPUTS (kept entirely separate from every earlier phase's folders)
----------------------------------------------------------------------------
phase5_statistics/tables/<dataset>/
    table_cv_confidence_intervals.csv     - t-based CI from saved CV stats
    table_test_set_confidence_intervals.csv - Wilson score CI on test accuracy
    table_cochrans_q.json                  - omnibus significance result
    table_pairwise_mcnemar.csv              - every pairwise comparison, Holm-adjusted
phase5_statistics/figures/<dataset>/
    confidence_intervals.png                - accuracy +/- CI per model (both flavors)
    mcnemar_significance_matrix.png          - which pairs are significantly different

COMMAND-LINE FLAGS
----------------------------------------------------------------------------
    --data-dir DIR        Folder holding the raw datasets (default: data)
    --results-dir DIR      Root folder with saved_models/ (default: results)
    --models-dir DIR        Root folder with scaler/encoder .joblib (default: models)
    --cv-folds N              Number of CV folds used in the ORIGINAL training
                                run, for the t-based CI (default 5 — matches
                                the default --cv in every earlier script;
                                override if you passed --cv N there)
    --alpha F                  Significance level (default 0.05)
    --wisdm-window N              Must match the value used originally (default 80)
    --wisdm-step N                  Must match the value used originally (default 40)
    --no-download                    Don't auto-download; expects data already present
"""

import os
import io
import json
import zipfile
import tarfile
import argparse
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from scipy.stats import chi2, t as t_dist

warnings.filterwarnings("ignore")

try:
    from scipy.stats import binomtest
    def _exact_binom_p(k, n):
        return binomtest(k, n, 0.5, alternative="two-sided").pvalue
except ImportError:  # older scipy
    from scipy.stats import binom_test
    def _exact_binom_p(k, n):
        return binom_test(k, n, 0.5, alternative="two-sided")

OUT_ROOT = "phase5_statistics"

UCI_HAR_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]
WISDM_URLS = [
    "https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_ar_latest.tar.gz",
]


# ===========================================================================
# 0. GENERIC HELPERS
# ===========================================================================

def _find_dataset_root(search_dir):
    if not os.path.isdir(search_dir):
        return None
    for root, dirs, files in os.walk(search_dir):
        if "features.txt" in files and "train" in dirs and "test" in dirs:
            return root
    return None


def _find_file(search_dir, predicate):
    if not os.path.isdir(search_dir):
        return None
    for root, _, files in os.walk(search_dir):
        for fname in files:
            if predicate(fname):
                return os.path.join(root, fname)
    return None


def _is_wisdm_raw_file(fname):
    f = fname.lower()
    return f.endswith("raw.txt") and "wisdm" in f


# ===========================================================================
# 1. LOCATE ALREADY-TRAINED ARTIFACTS (no training happens in this script)
# ===========================================================================

def find_dataset_artifacts(results_dir="results", models_dir="models"):
    """Auto-detects flat (single dataset) or nested (per-dataset subfolder)
    layouts for saved models + preprocessing artifacts, matching whatever
    main.py / har_tinyml.py / har_tinyml_phase2.py produced."""
    found = {}

    flat_saved = os.path.join(results_dir, "saved_models")
    flat_scaler = os.path.join(models_dir, "scaler.joblib")
    flat_encoder = os.path.join(models_dir, "label_encoder.joblib")
    flat_csv = os.path.join(results_dir, "comparison_results.csv")
    if os.path.isdir(flat_saved) and os.path.exists(flat_scaler) and os.path.exists(flat_encoder):
        found["Dataset"] = {"saved_models_dir": flat_saved, "scaler": flat_scaler,
                             "encoder": flat_encoder, "comparison_csv": flat_csv}

    if os.path.isdir(results_dir):
        for entry in sorted(os.listdir(results_dir)):
            sub_saved = os.path.join(results_dir, entry, "saved_models")
            sub_scaler = os.path.join(models_dir, entry, "scaler.joblib")
            sub_encoder = os.path.join(models_dir, entry, "label_encoder.joblib")
            sub_csv = os.path.join(results_dir, entry, "comparison_results.csv")
            if os.path.isdir(sub_saved) and os.path.exists(sub_scaler) and os.path.exists(sub_encoder):
                found[entry] = {"saved_models_dir": sub_saved, "scaler": sub_scaler,
                                 "encoder": sub_encoder, "comparison_csv": sub_csv}

    return found


def load_saved_models(saved_models_dir):
    models = {}
    for fname in sorted(os.listdir(saved_models_dir)):
        if fname.endswith(".joblib"):
            name = fname[:-len(".joblib")].replace("_", " ")
            models[name] = joblib.load(os.path.join(saved_models_dir, fname))
    return models


# ===========================================================================
# 2. TEST-SET-ONLY DATA LOADING (no training split is used by this script)
# ===========================================================================

def download_uci_har(data_dir="data"):
    target = os.path.join(data_dir, "UCI HAR Dataset")
    if os.path.isdir(target) and _find_dataset_root(target):
        return
    os.makedirs(data_dir, exist_ok=True)
    import urllib.request
    import shutil
    last_err = None
    for url in UCI_HAR_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                raw = resp.read()
            extract_dir = os.path.join(data_dir, "_uci_har_download_tmp")
            if os.path.isdir(extract_dir):
                shutil.rmtree(extract_dir)
            os.makedirs(extract_dir, exist_ok=True)
            with zipfile.ZipFile(io.BytesIO(raw)) as zf:
                zf.extractall(extract_dir)
            found_root = _find_dataset_root(extract_dir)
            if not found_root:
                for root, _, files in os.walk(extract_dir):
                    for fname in files:
                        if fname.lower().endswith(".zip"):
                            try:
                                with zipfile.ZipFile(os.path.join(root, fname)) as nzf:
                                    nzf.extractall(root)
                            except zipfile.BadZipFile:
                                continue
                found_root = _find_dataset_root(extract_dir)
            if not found_root:
                raise RuntimeError("Archive did not contain a recognizable UCI HAR structure.")
            if os.path.isdir(target):
                shutil.rmtree(target)
            shutil.move(found_root, target)
            shutil.rmtree(extract_dir, ignore_errors=True)
            return
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"[UCI HAR] Could not auto-download (last error: {last_err}).")


def load_uci_har_test_only(data_dir="data"):
    """Only reads test/ + the shared feature/activity label files — training
    data is never touched since this script doesn't train anything."""
    dataset_root = os.path.join(data_dir, "UCI HAR Dataset")
    if not os.path.isdir(dataset_root) or not os.path.exists(os.path.join(dataset_root, "features.txt")):
        found = _find_dataset_root(data_dir)
        if found:
            dataset_root = found
        else:
            raise FileNotFoundError(f"Could not find a valid UCI HAR Dataset under '{data_dir}'.")

    features = pd.read_csv(os.path.join(dataset_root, "features.txt"), sep=r"\s+", header=None,
                            names=["idx", "name"])
    seen, feature_names = {}, []
    for n in features["name"].tolist():
        if n in seen:
            seen[n] += 1
            feature_names.append(f"{n}_{seen[n]}")
        else:
            seen[n] = 0
            feature_names.append(n)

    labels_df = pd.read_csv(os.path.join(dataset_root, "activity_labels.txt"), sep=r"\s+",
                             header=None, names=["id", "activity"])
    activity_map = dict(zip(labels_df["id"], labels_df["activity"]))

    split_dir = os.path.join(dataset_root, "test")
    X_test = pd.read_csv(os.path.join(split_dir, "X_test.txt"), sep=r"\s+", header=None)
    X_test.columns = feature_names
    y_test = pd.read_csv(os.path.join(split_dir, "y_test.txt"), header=None, names=["Activity_Id"])
    y_test_labels = y_test["Activity_Id"].map(activity_map)

    return X_test.reset_index(drop=True), y_test_labels.reset_index(drop=True)


def download_wisdm(data_dir="data"):
    existing = _find_file(data_dir, _is_wisdm_raw_file)
    if existing:
        return existing
    os.makedirs(data_dir, exist_ok=True)
    import urllib.request
    last_err = None
    for url in WISDM_URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=90) as resp:
                raw = resp.read()
            extract_dir = os.path.join(data_dir, "WISDM_ar_v1.1")
            os.makedirs(extract_dir, exist_ok=True)
            with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tf:
                tf.extractall(extract_dir)
            raw_file = _find_file(extract_dir, _is_wisdm_raw_file)
            if not raw_file:
                raise RuntimeError("Could not locate the raw .txt file inside the WISDM archive.")
            return raw_file
        except Exception as e:
            last_err = e
            continue
    print(f"[WISDM] WARNING: could not auto-download (last error: {last_err}).")
    return None


def parse_wisdm_raw(path):
    with open(path, "r", errors="ignore") as f:
        content = f.read()
    content = content.replace("\n", "").replace("\r", "")
    records = content.split(";")
    rows = []
    for rec in records:
        rec = rec.strip().strip(",")
        if not rec:
            continue
        parts = rec.split(",")
        if len(parts) < 6:
            continue
        try:
            user = int(parts[0])
            activity = parts[1].strip()
            x, y, z = float(parts[3]), float(parts[4]), float(parts[5])
        except ValueError:
            continue
        rows.append((user, activity, x, y, z))
    return pd.DataFrame(rows, columns=["user", "activity", "x", "y", "z"])


def _make_windows(df, window_size, step):
    df = df.reset_index(drop=True)
    change = (df["user"] != df["user"].shift()) | (df["activity"] != df["activity"].shift())
    run_id = change.cumsum()
    windows = []
    for _, group in df.groupby(run_id):
        group = group.reset_index(drop=True)
        n = len(group)
        for start in range(0, max(n - window_size + 1, 0), step):
            windows.append(group.iloc[start:start + window_size])
    return windows


def _safe_corr(a, b):
    if np.std(a) == 0 or np.std(b) == 0:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def _extract_window_features(seg):
    feats = {}
    for axis in ("x", "y", "z"):
        vals = seg[axis].values.astype(float)
        feats[f"{axis}_mean"] = float(np.mean(vals))
        feats[f"{axis}_std"] = float(np.std(vals))
        feats[f"{axis}_min"] = float(np.min(vals))
        feats[f"{axis}_max"] = float(np.max(vals))
        feats[f"{axis}_range"] = feats[f"{axis}_max"] - feats[f"{axis}_min"]
        feats[f"{axis}_energy"] = float(np.mean(vals ** 2))
        feats[f"{axis}_mad"] = float(np.mean(np.abs(vals - np.mean(vals))))
        feats[f"{axis}_iqr"] = float(np.percentile(vals, 75) - np.percentile(vals, 25))
    x, y, z = seg["x"].values.astype(float), seg["y"].values.astype(float), seg["z"].values.astype(float)
    feats["corr_xy"] = _safe_corr(x, y)
    feats["corr_xz"] = _safe_corr(x, z)
    feats["corr_yz"] = _safe_corr(y, z)
    mag = np.sqrt(x ** 2 + y ** 2 + z ** 2)
    feats["mag_mean"] = float(np.mean(mag))
    feats["mag_std"] = float(np.std(mag))
    feats["mag_max"] = float(np.max(mag))
    feats["mag_min"] = float(np.min(mag))
    feats["activity"] = seg["activity"].iloc[0]
    feats["user"] = seg["user"].iloc[0]
    return feats


def load_wisdm_test_only(data_dir="data", window_size=80, step=40, test_size=0.2, random_state=42):
    """Reconstructs the SAME test split originally used (identical
    window_size/step/test_size/random_state -> identical train_test_split
    partition) so evaluation happens on the exact same held-out samples.
    This re-runs windowing (a data-processing step), not model training —
    no .fit() call happens anywhere in this function."""
    raw_path = _find_file(data_dir, _is_wisdm_raw_file)
    if not raw_path:
        raise FileNotFoundError(f"Could not find a WISDM raw .txt file under '{data_dir}'.")

    raw_df = parse_wisdm_raw(raw_path)
    if len(raw_df) == 0:
        raise ValueError(f"Parsed 0 readings from '{raw_path}'.")

    windows = _make_windows(raw_df, window_size, step)
    rows = [_extract_window_features(w) for w in windows if len(w) == window_size]
    feat_df = pd.DataFrame(rows)
    if len(feat_df) == 0:
        raise ValueError("No sliding windows could be extracted.")

    X = feat_df.drop(columns=["activity", "user"])
    y_labels = feat_df["activity"]

    _, X_test, _, y_test_labels = train_test_split(
        X, y_labels, test_size=test_size, random_state=random_state, stratify=y_labels
    )
    return X_test.reset_index(drop=True), y_test_labels.reset_index(drop=True)


# ===========================================================================
# 3. STATISTICAL TESTS
# ===========================================================================

def wilson_score_interval(n_correct, n_total, alpha=0.05):
    """Wilson score confidence interval for a binomial proportion (test-set
    accuracy) — the standard CI for exactly this situation, no resampling."""
    if n_total == 0:
        return np.nan, np.nan, np.nan
    from scipy.stats import norm
    z = norm.ppf(1 - alpha / 2)
    phat = n_correct / n_total
    denom = 1 + z ** 2 / n_total
    center = (phat + z ** 2 / (2 * n_total)) / denom
    half = (z * np.sqrt(phat * (1 - phat) / n_total + z ** 2 / (4 * n_total ** 2))) / denom
    return phat, max(0.0, center - half), min(1.0, center + half)


def cv_based_confidence_interval(mean_acc, std_acc, cv_folds, alpha=0.05):
    """Approximate CI from the k-fold CV mean/std Phase 1/2 already
    computed and saved — a t-distribution across the (small) number of
    folds, the standard approach when only fold-level summary stats
    (not raw per-fold scores) are available."""
    if cv_folds < 2 or pd.isna(std_acc):
        return mean_acc, mean_acc, mean_acc
    sem = std_acc / np.sqrt(cv_folds)
    h = sem * t_dist.ppf(1 - alpha / 2, cv_folds - 1)
    return mean_acc, mean_acc - h, mean_acc + h


def cochrans_q_test(correctness_matrix):
    """correctness_matrix: (n_samples, n_models) binary array, 1 = correct.
    Omnibus test across ALL models on the SAME test set — the appropriate
    single-test-set generalization of McNemar's test, playing the role
    Friedman's test would play if we had multiple independent CV folds."""
    n, k = correctness_matrix.shape
    col_sums = correctness_matrix.sum(axis=0)          # correct count per model
    row_sums = correctness_matrix.sum(axis=1)          # models-correct count per sample

    numerator = (k - 1) * (k * np.sum(col_sums ** 2) - np.sum(col_sums) ** 2)
    denominator = k * np.sum(row_sums) - np.sum(row_sums ** 2)
    if denominator == 0:
        return {"statistic": None, "p_value": None, "df": k - 1,
                "note": "All models agree on every sample — Cochran's Q is undefined (no variance to test)."}

    Q = numerator / denominator
    p_value = 1 - chi2.cdf(Q, df=k - 1)
    return {"statistic": round(float(Q), 4), "p_value": round(float(p_value), 6), "df": k - 1,
            "n_samples": int(n), "n_models": int(k)}


def mcnemar_exact_test(correct_a, correct_b):
    """Exact McNemar's test (binomial form) on two same-length binary
    correctness vectors for the same test samples. Returns (p_value, n01, n10)
    where n01 = A wrong & B correct, n10 = A correct & B wrong — only the
    discordant pairs carry information, which is McNemar's whole point."""
    n01 = int(np.sum((correct_a == 0) & (correct_b == 1)))
    n10 = int(np.sum((correct_a == 1) & (correct_b == 0)))
    n_discordant = n01 + n10
    if n_discordant == 0:
        return 1.0, n01, n10
    p = _exact_binom_p(min(n01, n10), n_discordant)
    return p, n01, n10


def pairwise_mcnemar_holm(predictions_correct, alpha=0.05):
    """predictions_correct: dict {model_name: binary correctness array}.
    Runs exact McNemar on every pair, then Holm-Bonferroni corrects across
    all pairs — the direct analogue of "pairwise Wilcoxon + Holm" but valid
    for a single shared test set instead of multiple CV folds."""
    names = list(predictions_correct.keys())
    pairs = list(combinations(names, 2))
    raw_p, meta = [], []
    for a, b in pairs:
        p, n01, n10 = mcnemar_exact_test(predictions_correct[a], predictions_correct[b])
        raw_p.append(p)
        meta.append((n01, n10))

    order = np.argsort(raw_p)
    m = len(raw_p)
    adjusted = [None] * m
    prev = 0.0
    for rank, idx in enumerate(order):
        adj = min(max((m - rank) * raw_p[idx], prev), 1.0)
        adjusted[idx] = adj
        prev = adj

    rows = []
    for (a, b), p_raw, p_adj, (n01, n10) in zip(pairs, raw_p, adjusted, meta):
        rows.append({"Model A": a, "Model B": b, "Discordant (A wrong/B right)": n01,
                     "Discordant (A right/B wrong)": n10,
                     "p-value (raw)": round(p_raw, 6), "p-value (Holm-adjusted)": round(p_adj, 6),
                     f"Significant (alpha={alpha})": p_adj < alpha})
    return pd.DataFrame(rows)


# ===========================================================================
# 4. FIGURES
# ===========================================================================

def plot_confidence_intervals(cv_ci_df, test_ci_df, out_path, label):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    for ax, df, title, mean_col, lo_col, hi_col in [
        (axes[0], cv_ci_df, f"CV-based CI (t-dist, k folds) — {label}",
         "Mean CV Accuracy", "CI Lower", "CI Upper"),
        (axes[1], test_ci_df, f"Test-set Wilson Score CI — {label}",
         "Test Accuracy", "CI Lower", "CI Upper"),
    ]:
        d = df.sort_values(mean_col, ascending=False)
        y_pos = np.arange(len(d))
        errs = np.array([d[mean_col] - d[lo_col], d[hi_col] - d[mean_col]])
        ax.errorbar(d[mean_col], y_pos, xerr=errs, fmt="o", color="#4C72B0", capsize=4)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(d["Model"])
        ax.set_xlabel("Accuracy")
        ax.set_title(title)
        ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_mcnemar_matrix(pairwise_df, model_names, out_path, label, alpha):
    n = len(model_names)
    idx = {m: i for i, m in enumerate(model_names)}
    sig_matrix = np.full((n, n), np.nan)
    p_matrix = np.full((n, n), np.nan)

    for _, row in pairwise_df.iterrows():
        i, j = idx[row["Model A"]], idx[row["Model B"]]
        p_adj = row["p-value (Holm-adjusted)"]
        p_matrix[i, j] = p_matrix[j, i] = p_adj
        sig_matrix[i, j] = sig_matrix[j, i] = 1.0 if p_adj < alpha else 0.0

    plt.figure(figsize=(7 + 0.3 * n, 6 + 0.3 * n))
    display = np.where(np.isnan(sig_matrix), 0.5, sig_matrix)
    plt.imshow(display, cmap="RdYlGn_r", vmin=0, vmax=1)
    for i in range(n):
        for j in range(n):
            if i == j:
                txt = "-"
            elif np.isnan(p_matrix[i, j]):
                txt = ""
            else:
                txt = f"{p_matrix[i, j]:.3f}"
            plt.text(j, i, txt, ha="center", va="center", fontsize=8)
    plt.xticks(range(n), model_names, rotation=45, ha="right")
    plt.yticks(range(n), model_names)
    plt.title(f"Pairwise McNemar Significance Matrix (Holm-adjusted p) — {label}\n"
              f"Red = significant at alpha={alpha}, Green = not significant")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


# ===========================================================================
# 5. PER-DATASET RUNNER
# ===========================================================================

def run_statistics_for_dataset(label, artifacts, args):
    print("\n" + "#" * 70)
    print(f"# PHASE 5 STATISTICAL VALIDATION — DATASET: {label}")
    print("#" * 70)

    print(f"[{label}] Loading saved models from '{artifacts['saved_models_dir']}' (inference only, no training)...")
    fitted_models = load_saved_models(artifacts["saved_models_dir"])
    if not fitted_models:
        raise RuntimeError(f"No .joblib models found in '{artifacts['saved_models_dir']}'.")
    print(f"[{label}] Loaded {len(fitted_models)} models: {list(fitted_models.keys())}")

    scaler = joblib.load(artifacts["scaler"])
    encoder = joblib.load(artifacts["encoder"])

    is_wisdm = "wisdm" in label.lower()
    print(f"[{label}] Loading test set only ({'WISDM' if is_wisdm else 'UCI HAR'} loader, no train split touched)...")
    if is_wisdm:
        if not args.no_download:
            download_wisdm(args.data_dir)
        X_test_raw, y_test_labels = load_wisdm_test_only(
            args.data_dir, window_size=args.wisdm_window, step=args.wisdm_step
        )
    else:
        if not args.no_download:
            download_uci_har(args.data_dir)
        X_test_raw, y_test_labels = load_uci_har_test_only(args.data_dir)

    X_test_scaled = scaler.transform(X_test_raw)
    y_test_enc = encoder.transform(y_test_labels)
    print(f"[{label}] Test set: {X_test_scaled.shape[0]} samples, {X_test_scaled.shape[1]} features.")

    tables_dir = os.path.join(OUT_ROOT, "tables", label)
    figures_dir = os.path.join(OUT_ROOT, "figures", label)
    os.makedirs(tables_dir, exist_ok=True)
    os.makedirs(figures_dir, exist_ok=True)

    # ---- Inference only: one .predict() call per model ----
    correctness = {}
    n_correct = {}
    for name, model in fitted_models.items():
        pred = model.predict(X_test_scaled)
        correctness[name] = (pred == y_test_enc).astype(int)
        n_correct[name] = int(correctness[name].sum())
    n_total = len(y_test_enc)

    # ---- CV-based CI (from Phase 1/2's already-computed cv_mean/cv_std) ----
    cv_ci_rows = []
    if artifacts["comparison_csv"] and os.path.exists(artifacts["comparison_csv"]):
        comp_df = pd.read_csv(artifacts["comparison_csv"])
        for _, r in comp_df.iterrows():
            if r["Model"] not in fitted_models:
                continue
            mean_acc, lo, hi = cv_based_confidence_interval(
                r.get("cv_mean_accuracy", np.nan), r.get("cv_std", np.nan), args.cv_folds, args.alpha
            )
            cv_ci_rows.append({"Model": r["Model"], "Mean CV Accuracy": round(mean_acc, 4),
                                "CI Lower": round(lo, 4), "CI Upper": round(hi, 4),
                                "CV Folds Assumed": args.cv_folds})
    else:
        print(f"[{label}] No comparison_results.csv found — skipping CV-based CI "
              "(test-set Wilson CI below is unaffected).")
    cv_ci_df = pd.DataFrame(cv_ci_rows) if cv_ci_rows else pd.DataFrame(
        columns=["Model", "Mean CV Accuracy", "CI Lower", "CI Upper", "CV Folds Assumed"])
    cv_ci_df.to_csv(os.path.join(tables_dir, "table_cv_confidence_intervals.csv"), index=False)

    # ---- Test-set Wilson score CI ----
    test_ci_rows = []
    for name in fitted_models:
        phat, lo, hi = wilson_score_interval(n_correct[name], n_total, args.alpha)
        test_ci_rows.append({"Model": name, "Test Accuracy": round(phat, 4),
                              "CI Lower": round(lo, 4), "CI Upper": round(hi, 4), "N": n_total})
    test_ci_df = pd.DataFrame(test_ci_rows).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)
    test_ci_df.to_csv(os.path.join(tables_dir, "table_test_set_confidence_intervals.csv"), index=False)

    print(f"\n[{label}] Test-set accuracy with 95% Wilson score CI:")
    print(test_ci_df.to_string(index=False))

    # ---- Cochran's Q (omnibus) ----
    model_names = list(fitted_models.keys())
    correctness_matrix = np.column_stack([correctness[m] for m in model_names])
    q_result = cochrans_q_test(correctness_matrix)
    with open(os.path.join(tables_dir, "table_cochrans_q.json"), "w") as f:
        json.dump(q_result, f, indent=2)
    print(f"\n[{label}] Cochran's Q test (omnibus — 'are ANY of these {len(model_names)} models "
          f"different?'): {q_result}")

    # ---- Pairwise McNemar + Holm (post-hoc, only meaningful if Q is significant,
    #      but we compute it regardless and let the significance column speak for itself) ----
    pairwise_df = pairwise_mcnemar_holm(correctness, alpha=args.alpha)
    pairwise_df.to_csv(os.path.join(tables_dir, "table_pairwise_mcnemar.csv"), index=False)
    print(f"\n[{label}] Pairwise McNemar tests (Holm-adjusted, alpha={args.alpha}):")
    print(pairwise_df.to_string(index=False))

    sig_pairs = pairwise_df[pairwise_df[f"Significant (alpha={args.alpha})"]]
    if q_result.get("p_value") is not None and q_result["p_value"] < args.alpha:
        print(f"\n[{label}] Cochran's Q is significant (p={q_result['p_value']:.4g} < {args.alpha}) — "
              "at least one model performs genuinely differently. Post-hoc pairwise McNemar "
              f"results above identify which pairs; {len(sig_pairs)} of {len(pairwise_df)} "
              "pairs remain significant after Holm correction.")
    else:
        print(f"\n[{label}] Cochran's Q is NOT significant — cannot claim any model is "
              "genuinely better than the others on this test set at the chosen alpha.")

    # ---- Figures ----
    if len(cv_ci_df) > 0:
        plot_confidence_intervals(cv_ci_df, test_ci_df,
                                   os.path.join(figures_dir, "confidence_intervals.png"), label)
    else:
        # test-only CI plot if no CV data available
        fig, ax = plt.subplots(figsize=(7, 5.5))
        d = test_ci_df.sort_values("Test Accuracy", ascending=False)
        y_pos = np.arange(len(d))
        errs = np.array([d["Test Accuracy"] - d["CI Lower"], d["CI Upper"] - d["Test Accuracy"]])
        ax.errorbar(d["Test Accuracy"], y_pos, xerr=errs, fmt="o", color="#4C72B0", capsize=4)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(d["Model"])
        ax.set_xlabel("Accuracy")
        ax.set_title(f"Test-set Wilson Score CI — {label}")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig(os.path.join(figures_dir, "confidence_intervals.png"), dpi=150)
        plt.close()

    plot_mcnemar_matrix(pairwise_df, model_names,
                         os.path.join(figures_dir, "mcnemar_significance_matrix.png"), label, args.alpha)

    print(f"\n[{label}] All Phase 5 outputs saved under {OUT_ROOT}/(tables|figures)/{label}/")
    return {"label": label, "cv_ci_df": cv_ci_df, "test_ci_df": test_ci_df,
            "cochrans_q": q_result, "pairwise_df": pairwise_df}


# ===========================================================================
# 6. MAIN
# ===========================================================================

def parse_args():
    p = argparse.ArgumentParser(
        description="TinyML HAR — Phase 5 statistical validation (standalone, no retraining)"
    )
    p.add_argument("--data-dir", default="data")
    p.add_argument("--results-dir", default="results")
    p.add_argument("--models-dir", default="models")
    p.add_argument("--cv-folds", type=int, default=5,
                    help="Number of CV folds used in the ORIGINAL training run (default 5)")
    p.add_argument("--alpha", type=float, default=0.05)
    p.add_argument("--wisdm-window", type=int, default=80)
    p.add_argument("--wisdm-step", type=int, default=40)
    p.add_argument("--no-download", action="store_true")
    args, _ = p.parse_known_args()
    return args


def main():
    args = parse_args()

    found = find_dataset_artifacts(args.results_dir, args.models_dir)
    if not found:
        print(
            f"No saved models found under '{args.results_dir}/**/saved_models/' with matching "
            f"scaler/encoder under '{args.models_dir}'.\n"
            "This script only runs inference on models you already trained — run main.py, "
            "har_tinyml.py, or har_tinyml_phase2.py first."
        )
        return

    print("=" * 70)
    print(f"PHASE 5: Statistical Validation — found {len(found)} dataset(s): {', '.join(found.keys())}")
    print("(loading saved models for inference only — nothing is retrained)")
    print("=" * 70)

    all_results = []
    for label, artifacts in found.items():
        try:
            all_results.append(run_statistics_for_dataset(label, artifacts, args))
        except Exception as e:
            print(f"\n[{label}] Skipping — {e}")

    print("\n" + "=" * 70)
    print("PHASE 5 COMPLETE")
    print("=" * 70)
    for res in all_results:
        q = res["cochrans_q"]
        verdict = "SIGNIFICANT differences exist among models" if (q.get("p_value") is not None and
                  q["p_value"] < args.alpha) else "no significant difference detected among models"
        print(f"[{res['label']}] Cochran's Q verdict: {verdict}")
    print(f"\nAll outputs under ./{OUT_ROOT}/ — separate folder, nothing retrained, "
          "nothing from earlier phases touched.")


if __name__ == "__main__":
    main()

PHASE 5: Statistical Validation — found 2 dataset(s): UCI_HAR, WISDM
(loading saved models for inference only — nothing is retrained)

######################################################################
# PHASE 5 STATISTICAL VALIDATION — DATASET: UCI_HAR
######################################################################
[UCI_HAR] Loading saved models from 'results/UCI_HAR/saved_models' (inference only, no training)...
[UCI_HAR] Loaded 7 models: ['Decision Tree', 'Gaussian Naive Bayes', 'KNN', 'Linear SVM', 'Logistic Regression', 'Random Forest', 'XGBoost']
[UCI_HAR] Loading test set only (UCI HAR loader, no train split touched)...
[UCI_HAR] Test set: 2947 samples, 561 features.

[UCI_HAR] Test-set accuracy with 95% Wilson score CI:
               Model  Test Accuracy  CI Lower  CI Upper    N
          Linear SVM         0.9620    0.9545    0.9683 2947
 Logistic Regression         0.9552    0.9471    0.9621 2947
             XGBoost         0.9477    0.9391    0.9552 2947
       